<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Block 1 — Load 19-column final dataset**

In [1]:
# ============================================================
# ML-05 / W05 — BLOCK 1
# LOAD FINAL 19-COLUMN FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/content/final_features_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 70)
print("FINAL FEATURE DATASET")
print("=" * 70)

print("Rows    :", f"{len(df):,}")
print("Columns :", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

print("\nShape:", df.shape)

FINAL FEATURE DATASET
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape: (2871202, 19)


**Block 2 — Missing indicators + 90-day eligibility**

In [2]:
# ============================================================
# BLOCK 2
# MISSING INDICATORS + BASIC DATA QUALITY
# ============================================================

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Remove meaningless missing indicator
# ------------------------------------------------------------

DROP_INDICATOR = "ai_other_missing"

if DROP_INDICATOR in df.columns:
    df = df.drop(columns=[DROP_INDICATOR])

print("=" * 70)
print("MISSING INDICATOR DECISION")
print("=" * 70)

print("Dropped:", DROP_INDICATOR)

remaining_indicators = [
    c for c in df.columns
    if c.endswith("_missing")
]

print("\nMissing indicators kept:")
for c in remaining_indicators:
    print("KEEP ->", c)

# ------------------------------------------------------------
# 2. Sort page-wise chronologically
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Check page history
# ------------------------------------------------------------

page_history = (
    df
    .groupby("content_hash_id")["month"]
    .nunique()
)

print("\n" + "=" * 70)
print("PAGE HISTORY")
print("=" * 70)

print(
    "Total pages:",
    f"{len(page_history):,}"
)

print(
    "Pages with < 3 months:",
    f"{(page_history < 3).sum():,}"
)

print(
    "Pages with >= 3 months:",
    f"{(page_history >= 3).sum():,}"
)

# ------------------------------------------------------------
# 4. Check monthly continuity
# ------------------------------------------------------------

df["previous_month"] = (
    df
    .groupby("content_hash_id")["month"]
    .shift(1)
)

df["month_gap"] = (
    (
        df["month"].dt.year
        - df["previous_month"].dt.year
    ) * 12
    +
    (
        df["month"].dt.month
        - df["previous_month"].dt.month
    )
)

gap_rows = df[
    df["month_gap"].notna()
    &
    (df["month_gap"] > 1)
]

print("\n" + "=" * 70)
print("MONTHLY CONTINUITY")
print("=" * 70)

print(
    "Rows with a month gap:",
    f"{len(gap_rows):,}"
)

print(
    "Pages affected:",
    f"{gap_rows['content_hash_id'].nunique():,}"
)

# helper columns no longer needed
df = df.drop(
    columns=[
        "previous_month",
        "month_gap"
    ]
)

MISSING INDICATOR DECISION
Dropped: ai_other_missing

Missing indicators kept:
KEEP -> gsc_avg_position_missing
KEEP -> ga4_total_engagement_sec_missing
KEEP -> sessions_organic_missing
KEEP -> sessions_ai_missing

PAGE HISTORY
Total pages: 427,292
Pages with < 3 months: 47,141
Pages with >= 3 months: 380,151

MONTHLY CONTINUITY
Rows with a month gap: 13,014
Pages affected: 11,525


**Block 3 — Feature impact + leakage checks**

In [3]:
# ============================================================
# BLOCK 3
# FEATURE AUDIT + LEAKAGE + VIF
# ============================================================

from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# 1. ID columns are NOT model features
# ------------------------------------------------------------

ID_COLUMNS = [
    "client_hash_id",
    "content_hash_id"
]

# ------------------------------------------------------------
# 2. Potential future/label-derived names
# ------------------------------------------------------------

future_keywords = [
    "future",
    "target",
    "label",
    "decay",
    "next_",
    "lead_"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in future_keywords
    ):
        suspicious_columns.append(col)

print("=" * 70)
print("LEAKAGE NAME CHECK")
print("=" * 70)

if suspicious_columns:
    print("Potentially suspicious columns:")
    for col in suspicious_columns:
        print("CHECK ->", col)
else:
    print("PASS: No obvious future/target-derived columns.")

# ------------------------------------------------------------
# 3. Constant columns
# ------------------------------------------------------------

feature_candidates = [
    c for c in df.columns
    if c not in ID_COLUMNS + ["month"]
]

constant_columns = [
    c for c in feature_candidates
    if df[c].nunique(dropna=False) <= 1
]

print("\n" + "=" * 70)
print("ZERO-VARIANCE CHECK")
print("=" * 70)

if constant_columns:
    for c in constant_columns:
        print("DROP ->", c)
else:
    print("PASS: No zero-variance columns.")

# ------------------------------------------------------------
# 4. Numeric feature list
# ------------------------------------------------------------

numeric_features = [
    c for c in feature_candidates
    if c not in constant_columns
    and pd.api.types.is_numeric_dtype(df[c])
]

print("\nNumeric model candidates:")
for c in numeric_features:
    print(" -", c)

# ------------------------------------------------------------
# 5. Correlation between current features
# ------------------------------------------------------------

corr_matrix = (
    df[numeric_features]
    .corr()
)

print("\n" + "=" * 70)
print("HIGH FEATURE-TO-FEATURE CORRELATION")
print("=" * 70)

high_corr_pairs = []

for i in range(len(numeric_features)):

    for j in range(i + 1, len(numeric_features)):

        a = numeric_features[i]
        b = numeric_features[j]

        corr = corr_matrix.loc[a, b]

        if abs(corr) >= 0.90:

            high_corr_pairs.append(
                (a, b, round(corr, 3))
            )

if high_corr_pairs:

    for pair in high_corr_pairs:
        print(pair)

else:

    print("No feature pairs with |correlation| >= 0.90")

# ------------------------------------------------------------
# 6. VIF sample
# ------------------------------------------------------------
# VIF on millions of rows is unnecessarily expensive.
# We use a reproducible sample for diagnostic purposes.

vif_sample = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sample(
        n=min(100_000, len(df)),
        random_state=42
    )
)

# Remove columns with zero variance inside sample
vif_features = [
    c for c in numeric_features
    if vif_sample[c].nunique() > 1
]

X_vif = vif_sample[vif_features]

vif_table = pd.DataFrame({
    "feature": vif_features,
    "VIF": [
        variance_inflation_factor(
            X_vif.values,
            i
        )
        for i in range(X_vif.shape[1])
    ]
})

vif_table = (
    vif_table
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("VIF TEST")
print("=" * 70)

display(vif_table)

LEAKAGE NAME CHECK
PASS: No obvious future/target-derived columns.

ZERO-VARIANCE CHECK
PASS: No zero-variance columns.

Numeric model candidates:
 - gsc_clicks
 - gsc_impressions
 - gsc_avg_position
 - ga4_total_engagement_sec
 - sessions_organic
 - sessions_ai
 - missing_count
 - gsc_avg_position_missing
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing
 - ctr
 - sec_per_click
 - ai_share
 - engagement_per_organic_session

HIGH FEATURE-TO-FEATURE CORRELATION
('gsc_clicks', 'sessions_organic', np.float64(0.978))
('missing_count', 'ga4_total_engagement_sec_missing', np.float64(0.968))
('missing_count', 'sessions_organic_missing', np.float64(0.968))
('missing_count', 'sessions_ai_missing', np.float64(0.968))
('ga4_total_engagement_sec_missing', 'sessions_organic_missing', np.float64(1.0))
('ga4_total_engagement_sec_missing', 'sessions_ai_missing', np.float64(1.0))
('sessions_organic_missing', 'sessions_ai_missing', np.float64(1.0))


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



VIF TEST


,feature,VIF
0,missing_count,inf
1,ga4_total_engagement_sec_missing,inf
2,gsc_avg_position_missing,inf
3,sessions_ai_missing,inf
4,sessions_organic_missing,inf
5,sessions_organic,4.518367
6,gsc_clicks,3.863913
7,gsc_impressions,2.184852
8,ga4_total_engagement_sec,2.176906
9,sessions_ai,1.924955


**Block 3.5: keep selected impactful  features after VIF and correlation test**

In [7]:
# ============================================================
# ML-05 → ML-06/07 PREPARATION
# FINAL FEATURE SET + 90-DAY ELIGIBILITY CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL FEATURE DATASET PREPARATION")
print("=" * 70)

# ============================================================
# 1. FIND THE CURRENT 19-COLUMN DATAFRAME
# ============================================================

# Try known dataframe names first
candidate_names = [
    "df_baseline",
    "df_model_base",
    "df_features_clean",
    "df_clean",
    "df"
]

source_df = None
source_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            source_df = obj.copy()
            source_name = name
            break

if source_df is None:
    raise NameError(
        "No suitable dataframe found. "
        "Please load your 19-column Parquet dataset first."
    )

print(f"Source dataframe used: {source_name}")
print(f"Rows before cleaning: {len(source_df):,}")
print(f"Columns before cleaning: {source_df.shape[1]}")

# ============================================================
# 2. START FROM SOURCE DATA
# ============================================================

df_model_base = source_df.copy()

# Convert month safely
df_model_base["month"] = pd.to_datetime(
    df_model_base["month"],
    errors="coerce"
)

# ============================================================
# 3. REMOVE REDUNDANT MISSINGNESS FEATURES
# ============================================================

remove_columns = [
    "missing_count",
    "ga4_total_engagement_sec_missing",
    "sessions_organic_missing",
    "sessions_ai_missing"
]

# Only remove columns that actually exist
remove_columns = [
    col
    for col in remove_columns
    if col in df_model_base.columns
]

df_model_base = df_model_base.drop(
    columns=remove_columns
)

print("\n" + "=" * 70)
print("REMOVED REDUNDANT FEATURES")
print("=" * 70)

if remove_columns:
    for col in remove_columns:
        print(" -", col)
else:
    print("None")

# ============================================================
# 4. REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "content_hash_id",
    "month"
]

missing_required = [
    col
    for col in required_columns
    if col not in df_model_base.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing: {missing_required}"
    )

# ============================================================
# 5. REMOVE INVALID MONTH ROWS
# ============================================================

before_month_filter = len(df_model_base)

df_model_base = df_model_base[
    df_model_base["month"].notna()
].copy()

removed_invalid_months = (
    before_month_filter -
    len(df_model_base)
)

print(
    f"\nRows removed because month was invalid: "
    f"{removed_invalid_months:,}"
)

# ============================================================
# 6. REMOVE DUPLICATE PAGE-MONTH RECORDS IF ANY
# ============================================================

duplicate_count = (
    df_model_base
    .duplicated(
        subset=["content_hash_id", "month"]
    )
    .sum()
)

print(
    f"Duplicate page-month rows found: "
    f"{duplicate_count:,}"
)

if duplicate_count > 0:
    df_model_base = (
        df_model_base
        .drop_duplicates(
            subset=["content_hash_id", "month"],
            keep="first"
        )
        .copy()
    )

# ============================================================
# 7. SORT PAGE HISTORIES CHRONOLOGICALLY
# ============================================================

df_model_base = (
    df_model_base
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 8. CALCULATE OBSERVED HISTORY
# ============================================================

first_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("max")
)

df_model_base["content_age_days"] = (
    last_month - first_month
).dt.days

# ============================================================
# 9. 90-DAY ELIGIBILITY
# ============================================================

eligible_mask = (
    df_model_base["content_age_days"] >= 90
)

df_model_eligible = (
    df_model_base.loc[eligible_mask]
    .copy()
)

removed_rows = (
    len(df_model_base) -
    len(df_model_eligible)
)

print("\n" + "=" * 70)
print("90-DAY ELIGIBILITY")
print("=" * 70)

print(
    f"Rows before 90-day filter : "
    f"{len(df_model_base):,}"
)

print(
    f"Rows after 90-day filter  : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Rows removed              : "
    f"{removed_rows:,}"
)

# ============================================================
# 10. PAGE-LEVEL ELIGIBILITY CHECK
# ============================================================

page_age = (
    df_model_eligible
    .groupby("content_hash_id")["content_age_days"]
    .max()
)

if len(page_age) > 0:

    print(
        f"\nEligible pages: "
        f"{len(page_age):,}"
    )

    print(
        f"Minimum eligible history: "
        f"{page_age.min()} days"
    )

    print(
        f"Pages with <90 days remaining: "
        f"{(page_age < 90).sum():,}"
    )

    # Safety check
    assert (
        page_age >= 90
    ).all(), (
        "ERROR: A page below 90 days remains."
    )

else:
    raise ValueError(
        "No pages remain after the 90-day eligibility filter."
    )

# ============================================================
# 11. FINAL CHRONOLOGICAL ORDER
# ============================================================

df_model_eligible = (
    df_model_eligible
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 12. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Final rows    : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Final columns : "
    f"{df_model_eligible.shape[1]}"
)

print(
    f"Final shape   : "
    f"{df_model_eligible.shape}"
)

print("\nRemaining columns:")

for i, col in enumerate(
    df_model_eligible.columns,
    start=1
):
    print(f"{i:2}. {col}")

# ============================================================
# 13. FINAL 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET — FIRST 5 ROWS")
print("=" * 70)

display(
    df_model_eligible.head(5)
)

# ============================================================
# 14. FINAL 90-DAY SAFETY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL SAFETY CHECK")
print("=" * 70)

print(
    "Minimum content_age_days:",
    df_model_eligible["content_age_days"].min()
)

print(
    "Rows below 90 days:",
    (
        df_model_eligible["content_age_days"] < 90
    ).sum()
)

assert (
    df_model_eligible["content_age_days"] >= 90
).all()

print("PASS: No page below the 90-day requirement.")

# ============================================================
# 15. SAVE FINAL INTERMEDIATE DATASET
# ============================================================

output_path = (
    "/content/final_model_eligible.parquet"
)

df_model_eligible.to_parquet(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("PARQUET SAVED")
print("=" * 70)

print(output_path)

print("\n" + "=" * 70)
print("PREPARATION COMPLETE")
print("=" * 70)

FINAL FEATURE DATASET PREPARATION
Source dataframe used: df_model_base
Rows before cleaning: 2,871,202
Columns before cleaning: 15

REMOVED REDUNDANT FEATURES
None

Rows removed because month was invalid: 0
Duplicate page-month rows found: 0

90-DAY ELIGIBILITY
Rows before 90-day filter : 2,871,202
Rows after 90-day filter  : 2,705,303
Rows removed              : 165,899

Eligible pages: 349,557
Minimum eligible history: 92 days
Pages with <90 days remaining: 0

FINAL DATASET
Final rows    : 2,705,303
Final columns : 15
Final shape   : (2705303, 15)

Remaining columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. gsc_avg_position_missing
11. ctr
12. sec_per_click
13. ai_share
14. engagement_per_organic_session
15. content_age_days

FINAL DATASET — FIRST 5 ROWS


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,gsc_avg_position_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session,content_age_days
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0.006849,0.0,0.0,0.0,457
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457



FINAL SAFETY CHECK
Minimum content_age_days: 92
Rows below 90 days: 0
PASS: No page below the 90-day requirement.

PARQUET SAVED
/content/final_model_eligible.parquet

PREPARATION COMPLETE


**BLOCK 4 — ROLLING 90-DAY FEATURES + 30-DAY BASELINE SIGNAL**

In [8]:
# ============================================================
# BLOCK 4 — ROLLING 90-DAY FEATURES
#
# CURRENT 3 MONTHS = MODEL INPUT HISTORY
# NEXT 3 MONTHS    = TARGET GENERATION DATA ONLY
#
# ALSO PRESERVED:
#   gsc_impressions_prev_30d
#   gsc_impressions_last_30d
#
# These two columns are required later for the
# Early Drop Baseline comparison.
#
# IMPORTANT:
# NO target / target_label is created in this block.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4 — ROLLING 90-DAY FEATURES + EARLY DROP SIGNAL")
print("=" * 80)

# ============================================================
# 1. SOURCE DATA
# ============================================================

if "df_model_eligible" not in globals():
    raise NameError(
        "df_model_eligible not found. "
        "Run the 90-day eligibility preparation block first."
    )

df_roll = df_model_eligible.copy()

print(f"Source rows : {len(df_roll):,}")
print(f"Source cols : {df_roll.shape[1]}")

# ============================================================
# 2. DATE CLEANING
# ============================================================

df_roll["month"] = pd.to_datetime(
    df_roll["month"],
    errors="coerce"
)

df_roll = df_roll.dropna(
    subset=["content_hash_id", "month"]
).copy()

# ============================================================
# 3. DUPLICATE PAGE-MONTH CLEANUP
# ============================================================

before_dup = len(df_roll)

df_roll = (
    df_roll
    .drop_duplicates(
        subset=["content_hash_id", "month"],
        keep="first"
    )
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

print(
    f"Duplicate rows removed: "
    f"{before_dup - len(df_roll):,}"
)

# ============================================================
# 4. MONTH PERIOD
# ============================================================

df_roll["month_period"] = (
    df_roll["month"].dt.to_period("M")
)

# ============================================================
# 5. REQUIRED RAW FEATURES
# ============================================================

required_features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "gsc_avg_position_missing",
    "ctr",
    "sec_per_click",
    "ai_share",
    "engagement_per_organic_session"
]

missing_features = [
    col
    for col in required_features
    if col not in df_roll.columns
]

if missing_features:
    raise KeyError(
        "Required raw feature columns are missing:\n"
        + "\n".join(missing_features)
    )

# ============================================================
# 6. VERIFY 30-DAY BASELINE SOURCE
#
# We need monthly impressions.
#
# For each rolling row:
#
# prev_30d = first month of current 3-month window
# last_30d = latest month of current 3-month window
#
# This keeps the Early Drop signal aligned with
# the same rolling-window observation.
# ============================================================

print("\n" + "=" * 80)
print("30-DAY EARLY DROP BASELINE")
print("=" * 80)

print(
    "Baseline definition:"
)
print(
    "gsc_impressions_last_30d < gsc_impressions_prev_30d"
)

# ============================================================
# 7. CONSECUTIVE 6-MONTH WINDOWS
#
# Current:
#   i     i+1     i+2
#
# Future:
#   i+3   i+4     i+5
#
# All six months must be consecutive.
# ============================================================

page = df_roll["content_hash_id"]
period = df_roll["month_period"]

valid_6m = (
    page.eq(page.shift(-1))
    & page.eq(page.shift(-2))
    & page.eq(page.shift(-3))
    & page.eq(page.shift(-4))
    & page.eq(page.shift(-5))

    & period.add(1).eq(period.shift(-1))
    & period.add(2).eq(period.shift(-2))
    & period.add(3).eq(period.shift(-3))
    & period.add(4).eq(period.shift(-4))
    & period.add(5).eq(period.shift(-5))
)

valid_idx = np.flatnonzero(
    valid_6m.to_numpy()
)

print(
    f"Valid 6-month windows: "
    f"{len(valid_idx):,}"
)

if len(valid_idx) == 0:
    raise ValueError(
        "No valid 6-month rolling windows found."
    )

# ============================================================
# 8. OUTPUT DATAFRAME
# ============================================================

out = pd.DataFrame(
    index=np.arange(len(valid_idx))
)

# ============================================================
# 9. METADATA
# ============================================================

out["content_hash_id"] = (
    df_roll[
        "content_hash_id"
    ]
    .iloc[valid_idx]
    .to_numpy()
)

if "client_hash_id" in df_roll.columns:

    out["client_hash_id"] = (
        df_roll[
            "client_hash_id"
        ]
        .iloc[valid_idx]
        .to_numpy()
    )

# Current 90-day window
out["window_start"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx]
    .to_numpy()
)

out["window_end"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx + 2]
    .to_numpy()
)

# Future 90-day window
out["future_start"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx + 3]
    .to_numpy()
)

out["future_end"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx + 5]
    .to_numpy()
)

# ============================================================
# 10. CURRENT 3-MONTH MODEL FEATURES
# ============================================================

print("\n" + "=" * 80)
print("CREATING CURRENT 90-DAY FEATURES")
print("=" * 80)

for col in required_features:

    values = pd.to_numeric(
        df_roll[col],
        errors="coerce"
    ).to_numpy()

    v0 = values[valid_idx]
    v1 = values[valid_idx + 1]
    v2 = values[valid_idx + 2]

    # 3-month mean
    out[
        f"{col}_mean_3m"
    ] = (
        v0 + v1 + v2
    ) / 3

    # Latest/current month
    out[
        f"{col}_last"
    ] = v2

# ============================================================
# 11. CURRENT 90-DAY IMPRESSIONS
# ============================================================

imp = pd.to_numeric(
    df_roll["gsc_impressions"],
    errors="coerce"
).to_numpy()

current_imp_3m = (
    imp[valid_idx]
    + imp[valid_idx + 1]
    + imp[valid_idx + 2]
) / 3

out["current_imp_3m"] = (
    current_imp_3m
)

# ============================================================
# 12. EARLY DROP BASELINE COLUMNS
#
# IMPORTANT:
#
# prev_30d = first month of current 90-day window
# last_30d = last month of current 90-day window
#
# This reproduces the intended:
#
# last_30d < prev_30d
# ============================================================

out["gsc_impressions_prev_30d"] = (
    imp[valid_idx]
)

out["gsc_impressions_last_30d"] = (
    imp[valid_idx + 2]
)

# ============================================================
# 13. EARLY DROP SIGNAL
# ============================================================

prev_30d = pd.to_numeric(
    out[
        "gsc_impressions_prev_30d"
    ],
    errors="coerce"
)

last_30d = pd.to_numeric(
    out[
        "gsc_impressions_last_30d"
    ],
    errors="coerce"
)

out["early_drop_signal"] = (
    last_30d < prev_30d
)

# ============================================================
# 14. FUTURE IMPRESSIONS
#
# Target-generation information only.
# ============================================================

future_imp_3m = (
    imp[valid_idx + 3]
    + imp[valid_idx + 4]
    + imp[valid_idx + 5]
) / 3

out["future_imp_3m"] = (
    future_imp_3m
)

# ============================================================
# 15. FUTURE IMPRESSION CHANGE %
# ============================================================

valid_change = (
    np.isfinite(current_imp_3m)
    & np.isfinite(future_imp_3m)
    & (current_imp_3m > 0)
)

out["future_impression_change_pct"] = (
    np.nan
)

out.loc[
    valid_change,
    "future_impression_change_pct"
] = (
    (
        future_imp_3m[valid_change]
        - current_imp_3m[valid_change]
    )
    / current_imp_3m[valid_change]
) * 100

# ============================================================
# 16. REMOVE INVALID FUTURE CHANGE ROWS
# ============================================================

before_target_validity = len(out)

out = out[
    out[
        "future_impression_change_pct"
    ].notna()
].copy()

print(
    f"Rows removed due to invalid "
    f"future change: "
    f"{before_target_validity - len(out):,}"
)

# ============================================================
# 17. SAFETY CHECK
# ============================================================

assert (
    out[
        "future_impression_change_pct"
    ].notna().all()
)

assert (
    out[
        "gsc_impressions_prev_30d"
    ].notna().all()
)

assert (
    out[
        "gsc_impressions_last_30d"
    ].notna().all()
)

# ============================================================
# 18. COLUMN SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ROLLING WINDOW DATASET")
print("=" * 80)

print(
    f"Rows    : {len(out):,}"
)

print(
    f"Columns : {out.shape[1]}"
)

print("\nColumns:")

for i, col in enumerate(
    out.columns,
    start=1
):
    print(
        f"{i:2}. {col}"
    )

# ============================================================
# 19. EARLY DROP BASELINE SUMMARY
# ============================================================

early_drop_count = int(
    out["early_drop_signal"].sum()
)

early_drop_pct = (
    early_drop_count
    / len(out)
    * 100
)

print("\n" + "=" * 80)
print("EARLY DROP BASELINE SIGNAL")
print("=" * 80)

print(
    f"Early Drop pages : "
    f"{early_drop_count:,}"
)

print(
    f"Early Drop rate  : "
    f"{early_drop_pct:.2f}%"
)

print(
    "\nRule:"
)

print(
    "gsc_impressions_last_30d "
    "< gsc_impressions_prev_30d"
)

# ============================================================
# 20. PREVIEW
# ============================================================

print("\n" + "=" * 80)
print("ROLLING 90-DAY DATASET PREVIEW")
print("=" * 80)

display(
    out.head(10)
)

# ============================================================
# 21. SAVE BLOCK 4 DATASET
# ============================================================

rolling90_features_path = (
    "/content/rolling90_features.parquet"
)

out.to_parquet(
    rolling90_features_path,
    index=False
)

print("\n" + "=" * 80)
print("BLOCK 4 SAVED")
print("=" * 80)

print(
    rolling90_features_path
)

print("\n✓ Current 90-day features created.")
print("✓ Future 90-day information retained for target generation.")
print("✓ Early Drop baseline columns retained.")
print("✓ No target labels created yet.")

BLOCK 4 — ROLLING 90-DAY FEATURES + EARLY DROP SIGNAL
Source rows : 2,705,303
Source cols : 15
Duplicate rows removed: 0

30-DAY EARLY DROP BASELINE
Baseline definition:
gsc_impressions_last_30d < gsc_impressions_prev_30d
Valid 6-month windows: 978,801

CREATING CURRENT 90-DAY FEATURES
Rows removed due to invalid future change: 351,965

ROLLING WINDOW DATASET
Rows    : 626,836
Columns : 34

Columns:
 1. content_hash_id
 2. client_hash_id
 3. window_start
 4. window_end
 5. future_start
 6. future_end
 7. gsc_clicks_mean_3m
 8. gsc_clicks_last
 9. gsc_impressions_mean_3m
10. gsc_impressions_last
11. gsc_avg_position_mean_3m
12. gsc_avg_position_last
13. ga4_total_engagement_sec_mean_3m
14. ga4_total_engagement_sec_last
15. sessions_organic_mean_3m
16. sessions_organic_last
17. sessions_ai_mean_3m
18. sessions_ai_last
19. gsc_avg_position_missing_mean_3m
20. gsc_avg_position_missing_last
21. ctr_mean_3m
22. ctr_last
23. sec_per_click_mean_3m
24. sec_per_click_last
25. ai_share_mean_3m
26

,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,gsc_impressions_prev_30d,gsc_impressions_last_30d,early_drop_signal,future_imp_3m,future_impression_change_pct
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,0.0,0.0,136.666667,7.0,257.0,False,378.333333,176.829268
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,0.0,0.0,180.666667,146.0,139.0,True,506.666667,180.442804
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,0.0,0.0,216.666667,257.0,254.0,True,487.666667,125.076923
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,0.0,0.0,378.333333,139.0,742.0,False,280.333333,-25.903084
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,0.0,0.0,506.666667,254.0,524.0,False,167.000000,-67.039474
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.666667,0.0,487.666667,197.0,...,0.0,0.0,0.0,0.0,487.666667,742.0,197.0,True,106.333333,-78.195489
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.333333,0.0,280.333333,120.0,...,0.0,0.0,0.0,0.0,280.333333,524.0,120.0,True,74.333333,-73.483948
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,167.000000,184.0,...,0.0,0.0,0.0,0.0,167.000000,197.0,184.0,True,41.666667,-75.049900
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.000000,0.0,106.333333,15.0,...,0.0,0.0,0.0,0.0,106.333333,120.0,15.0,True,63.666667,-40.125392
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,2026-02-01,2026-03-01,2026-05-01,0.000000,0.0,74.333333,24.0,...,0.0,0.0,0.0,0.0,74.333333,184.0,24.0,True,82.666667,11.210762



BLOCK 4 SAVED
/content/rolling90_features.parquet

✓ Current 90-day features created.
✓ Future 90-day information retained for target generation.
✓ Early Drop baseline columns retained.
✓ No target labels created yet.


**BLOCK 4.5 — TARGET THRESHOLD VALIDATION**

Where Target does not apply only see future_impression_change_pct and compare candidate thresholds  


In [9]:
# ============================================================
# BLOCK 4.5 — TARGET THRESHOLD VALIDATION
#
# NO TARGET LABEL IS CREATED HERE.
#
# Purpose:
# Select a meaningful threshold ONLY on the
# rolling-window observations.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4.5 — TARGET THRESHOLD VALIDATION")
print("=" * 80)

if "out" not in globals():
    raise NameError(
        "Rolling-window dataframe 'out' not found. "
        "Run Block 4 first."
    )

df_threshold = out.copy()

change = pd.to_numeric(
    df_threshold[
        "future_impression_change_pct"
    ],
    errors="coerce"
).dropna()

print(
    f"\nValid rolling windows: "
    f"{len(change):,}"
)

# ============================================================
# 1. DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("FUTURE IMPRESSION CHANGE DISTRIBUTION")
print("=" * 80)

distribution = pd.DataFrame({
    "statistic": [
        "Minimum",
        "5th percentile",
        "10th percentile",
        "25th percentile",
        "Median",
        "75th percentile",
        "90th percentile",
        "95th percentile",
        "Maximum"
    ],
    "change_pct": [
        change.min(),
        change.quantile(.05),
        change.quantile(.10),
        change.quantile(.25),
        change.median(),
        change.quantile(.75),
        change.quantile(.90),
        change.quantile(.95),
        change.max()
    ]
})

distribution["change_pct"] = (
    distribution["change_pct"]
    .round(2)
)

display(distribution)

# ============================================================
# 2. CANDIDATE THRESHOLDS
# ============================================================

symmetric_rules = [
    (-10, 10),
    (-15, 15),
    (-20, 20),
    (-25, 25),
    (-30, 30),
    (-40, 40),
    (-50, 50)
]

asymmetric_rules = [
    (-20, 50),
    (-25, 50),
    (-30, 50),
    (-40, 50),
    (-50, 50),

    (-30, 60),
    (-40, 60),
    (-50, 60),

    (-30, 75),
    (-40, 75),
    (-50, 75),

    (-50, 100)
]

all_rules = (
    symmetric_rules
    + asymmetric_rules
)

# Remove duplicate rules
all_rules = list(
    dict.fromkeys(all_rules)
)

# ============================================================
# 3. BALANCE SCORE
#
# Higher score = more balanced classes.
#
# This is ONLY a screening metric.
# We do NOT automatically select the highest score.
# ============================================================

results = []

for down_threshold, up_threshold in all_rules:

    down_mask = (
        change <= down_threshold
    )

    flat_mask = (
        (change > down_threshold)
        & (change < up_threshold)
    )

    up_mask = (
        change >= up_threshold
    )

    down_pct = (
        down_mask.mean() * 100
    )

    flat_pct = (
        flat_mask.mean() * 100
    )

    up_pct = (
        up_mask.mean() * 100
    )

    # Simple balance score:
    # maximum possible when all three are ~33.33%
    balance_score = (
        100
        - (
            abs(down_pct - 33.33)
            + abs(flat_pct - 33.33)
            + abs(up_pct - 33.33)
        )
    )

    results.append({
        "rule": (
            f"{down_threshold}% / "
            f"+{up_threshold}%"
        ),
        "down_pct": round(down_pct, 2),
        "flat_pct": round(flat_pct, 2),
        "up_pct": round(up_pct, 2),
        "balance_score": round(
            balance_score,
            2
        )
    })

threshold_results = pd.DataFrame(
    results
)

# ============================================================
# 4. SYMMETRIC RULES
# ============================================================

print("\n" + "=" * 80)
print("SYMMETRIC THRESHOLD COMPARISON")
print("=" * 80)

symmetric_labels = [
    f"±{x}%"
    for x in [10, 15, 20, 25, 30, 40, 50]
]

symmetric_table = threshold_results[
    threshold_results["rule"].isin(
        symmetric_labels
    )
].copy()

display(
    symmetric_table.reset_index(drop=True)
)

# ============================================================
# 5. ASYMMETRIC RULES
# ============================================================

print("\n" + "=" * 80)
print("ASYMMETRIC THRESHOLD COMPARISON")
print("=" * 80)

asymmetric_table = threshold_results[
    ~threshold_results["rule"].isin(
        symmetric_labels
    )
].copy()

display(
    asymmetric_table
    .sort_values(
        "balance_score",
        ascending=False
    )
    .reset_index(drop=True)
)

# ============================================================
# 6. PROJECT WORKING RULE
#
# Based on your validated rule:
#
# <= -30%  = DOWN
# > -30% and < +50% = FLAT
# >= +50% = UP
# ============================================================

recommended_down = -30
recommended_up = 50

recommended_mask = (
    change <= recommended_down
)

recommended_flat = (
    (change > recommended_down)
    & (change < recommended_up)
)

recommended_up_mask = (
    change >= recommended_up
)

recommended_result = pd.DataFrame([{
    "rule": "PROJECT RULE: -30% / +50%",
    "down_pct": round(
        recommended_mask.mean() * 100,
        2
    ),
    "flat_pct": round(
        recommended_flat.mean() * 100,
        2
    ),
    "up_pct": round(
        recommended_up_mask.mean() * 100,
        2
    )
}])

print("\n" + "=" * 80)
print("PROJECT TARGETING RULE")
print("=" * 80)

display(
    recommended_result
)

print(
    "\nRecommended working rule:"
)

print(
    "DOWN : change <= -30%"
)

print(
    "FLAT : -30% < change < +50%"
)

print(
    "UP   : change >= +50%"
)

# ============================================================
# 7. TARGET ENCODING VERIFICATION
# ============================================================

print("\n" + "=" * 80)
print("TARGET ENCODING PLAN")
print("=" * 80)

print("0 = DOWN")
print("1 = FLAT")
print("2 = UP")

print(
    "\n✓ Threshold validation complete."
)

print(
    "✓ No target column has been created."
)

print(
    "✓ The rolling-window dataframe remains unchanged."
)

BLOCK 4.5 — TARGET THRESHOLD VALIDATION

Valid rolling windows: 626,836

FUTURE IMPRESSION CHANGE DISTRIBUTION


,statistic,change_pct
0,Minimum,-100.00
1,5th percentile,-100.00
2,10th percentile,-90.00
3,25th percentile,-50.16
4,Median,19.57
5,75th percentile,150.00
6,90th percentile,442.13
7,95th percentile,909.09
8,Maximum,2306500.00



SYMMETRIC THRESHOLD COMPARISON


,rule,down_pct,flat_pct,up_pct,balance_score



ASYMMETRIC THRESHOLD COMPARISON


,rule,down_pct,flat_pct,up_pct,balance_score
0,-30% / +75%,32.93,30.48,36.59,93.48
1,-40% / +75%,29.37,34.03,36.59,92.07
2,-40% / +60%,29.37,30.96,39.66,87.34
3,-30% / +60%,32.93,27.41,39.66,87.34
4,-50% / +60%,25.98,34.35,39.66,85.30
5,-50% / +75%,25.98,37.42,36.59,85.30
6,-50% / +100%,25.98,41.66,32.36,83.35
7,-40% / +50%,29.37,28.55,42.08,82.50
8,-50% / +50%,25.98,31.93,42.08,82.50
9,-30% / +50%,32.93,24.99,42.08,82.50



PROJECT TARGETING RULE


,rule,down_pct,flat_pct,up_pct
0,PROJECT RULE: -30% / +50%,32.93,24.99,42.08



Recommended working rule:
DOWN : change <= -30%
FLAT : -30% < change < +50%
UP   : change >= +50%

TARGET ENCODING PLAN
0 = DOWN
1 = FLAT
2 = UP

✓ Threshold validation complete.
✓ No target column has been created.
✓ The rolling-window dataframe remains unchanged.


**BLOCK 4.6 — APPLY FROZEN TARGET + SAVE FINAL ROLLING90WINDOW**

In [10]:
# ============================================================
# BLOCK 4.6 — APPLY FROZEN TARGET
#
# FINAL ROLLING 90-DAY DATASET
#
# Target:
#   0 = DOWN
#   1 = FLAT
#   2 = UP
#
# Frozen thresholds:
#   <= -30%          -> DOWN
#   > -30% & < +50% -> FLAT
#   >= +50%         -> UP
#
# IMPORTANT:
# All metadata
# All current-window features
# All future target-generation columns
# Target
# Target label
# are retained.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4.6 — FINAL ROLLING 90-DAY DATASET")
print("=" * 80)

if "out" not in globals():
    raise NameError(
        "Rolling-window dataframe 'out' not found. "
        "Run Block 4 first."
    )

finalrolling90 = out.copy()

# ============================================================
# 1. FROZEN TARGET THRESHOLDS
# ============================================================

DOWN_THRESHOLD = -30
UP_THRESHOLD = 50

print("\n" + "=" * 80)
print("FROZEN TARGET RULE")
print("=" * 80)

print(
    f"DOWN : change <= {DOWN_THRESHOLD}%"
)

print(
    f"FLAT : {DOWN_THRESHOLD}% < change < "
    f"+{UP_THRESHOLD}%"
)

print(
    f"UP   : change >= +{UP_THRESHOLD}%"
)

# ============================================================
# 2. CREATE NUMERIC TARGET
# ============================================================

change = pd.to_numeric(
    finalrolling90[
        "future_impression_change_pct"
    ],
    errors="coerce"
)

if change.isna().any():
    raise ValueError(
        "NaN values found in future_impression_change_pct."
    )

finalrolling90["target"] = np.select(
    [
        change <= DOWN_THRESHOLD,
        change >= UP_THRESHOLD
    ],
    [
        0,
        2
    ],
    default=1
).astype("int8")

# ============================================================
# 3. CREATE HUMAN-READABLE TARGET LABEL
# ============================================================

label_map = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

finalrolling90["target_label"] = (
    finalrolling90["target"]
    .map(label_map)
)

# ============================================================
# 4. TARGET ENCODING SAFETY CHECK
# ============================================================

print("\n" + "=" * 80)
print("TARGET ENCODING")
print("=" * 80)

print("0 = DOWN")
print("1 = FLAT")
print("2 = UP")

assert set(
    finalrolling90["target"].unique()
).issubset({0, 1, 2})

assert (
    finalrolling90["target_label"]
    .notna()
    .all()
)

assert (
    finalrolling90.loc[
        finalrolling90["target"] == 0,
        "target_label"
    ].eq("DOWN").all()
)

assert (
    finalrolling90.loc[
        finalrolling90["target"] == 1,
        "target_label"
    ].eq("FLAT").all()
)

assert (
    finalrolling90.loc[
        finalrolling90["target"] == 2,
        "target_label"
    ].eq("UP").all()
)

print("✓ Encoding confirmed.")

# ============================================================
# 5. BOUNDARY SANITY CHECK
# ============================================================

down_boundary_ok = (
    finalrolling90.loc[
        finalrolling90["target"] == 0,
        "future_impression_change_pct"
    ] <= -30
).all()

flat_values = finalrolling90.loc[
    finalrolling90["target"] == 1,
    "future_impression_change_pct"
]

flat_boundary_ok = (
    (flat_values > -30)
    & (flat_values < 50)
).all()

up_boundary_ok = (
    finalrolling90.loc[
        finalrolling90["target"] == 2,
        "future_impression_change_pct"
    ] >= 50
).all()

assert down_boundary_ok
assert flat_boundary_ok
assert up_boundary_ok

print("\n" + "=" * 80)
print("BOUNDARY SANITY CHECK")
print("=" * 80)

print(
    "✓ DOWN contains only change <= -30%"
)

print(
    "✓ FLAT contains only -30% < change < +50%"
)

print(
    "✓ UP contains only change >= +50%"
)

# ============================================================
# 6. FINAL TARGET DISTRIBUTION
# ============================================================

target_summary = (
    finalrolling90
    .groupby(
        ["target", "target_label"],
        sort=True
    )
    .size()
    .reset_index(
        name="count"
    )
)

target_summary["percentage"] = (
    target_summary["count"]
    / len(finalrolling90)
    * 100
).round(2)

print("\n" + "=" * 80)
print("FINAL TARGET DISTRIBUTION")
print("=" * 80)

display(
    target_summary
)

# ============================================================
# 7. EARLY DROP BASELINE SUMMARY
# ============================================================

early_drop_count = int(
    finalrolling90[
        "early_drop_signal"
    ].sum()
)

early_drop_pct = (
    early_drop_count
    / len(finalrolling90)
    * 100
)

print("\n" + "=" * 80)
print("EARLY DROP BASELINE")
print("=" * 80)

print(
    f"Early Drop signals : "
    f"{early_drop_count:,}"
)

print(
    f"Early Drop rate    : "
    f"{early_drop_pct:.2f}%"
)

print(
    "\nRule:"
)

print(
    "gsc_impressions_last_30d "
    "< gsc_impressions_prev_30d"
)

# ============================================================
# 8. FINAL COLUMN INVENTORY
# ============================================================

print("\n" + "=" * 80)
print("FINAL ROLLING90WINDOW DATASET")
print("=" * 80)

print(
    f"Rows    : "
    f"{len(finalrolling90):,}"
)

print(
    f"Columns : "
    f"{finalrolling90.shape[1]}"
)

print("\nAll columns:")

for i, col in enumerate(
    finalrolling90.columns,
    start=1
):
    print(
        f"{i:2}. {col}"
    )

# ============================================================
# 9. PREVIEW
# ============================================================

print("\n" + "=" * 80)
print("FINAL DATASET PREVIEW")
print("=" * 80)

display(
    finalrolling90.head(10)
)

# ============================================================
# 10. FINAL SAFETY CHECKS
# ============================================================

assert (
    finalrolling90[
        "target"
    ].notna().all()
)

assert (
    finalrolling90[
        "target_label"
    ].notna().all()
)

assert (
    finalrolling90[
        "future_imp_3m"
    ].notna().all()
)

assert (
    finalrolling90[
        "future_impression_change_pct"
    ].notna().all()
)

assert (
    "future_start"
    in finalrolling90.columns
)

assert (
    "future_end"
    in finalrolling90.columns
)

assert (
    "gsc_impressions_prev_30d"
    in finalrolling90.columns
)

assert (
    "gsc_impressions_last_30d"
    in finalrolling90.columns
)

assert (
    "early_drop_signal"
    in finalrolling90.columns
)

# ============================================================
# 11. SAVE FINAL DATASET
# ============================================================

finalrolling90_path = (
    "/content/finalrolling90window.parquet"
)

finalrolling90.to_parquet(
    finalrolling90_path,
    index=False
)

print("\n" + "=" * 80)
print("FINAL ROLLING90WINDOW SAVED")
print("=" * 80)

print(
    finalrolling90_path
)

print("\n" + "=" * 80)
print("BLOCK 4.6 COMPLETE")
print("=" * 80)

print(
    "✓ 90-day rolling features retained"
)

print(
    "✓ Future 90-day target-generation data retained"
)

print(
    "✓ Previous 30-day impressions retained"
)

print(
    "✓ Last 30-day impressions retained"
)

print(
    "✓ Early Drop baseline signal retained"
)

print(
    "✓ Target created: 0=DOWN, 1=FLAT, 2=UP"
)

print(
    "✓ Target labels created"
)

print(
    "✓ Final parquet saved"
)

BLOCK 4.6 — FINAL ROLLING 90-DAY DATASET

FROZEN TARGET RULE
DOWN : change <= -30%
FLAT : -30% < change < +50%
UP   : change >= +50%

TARGET ENCODING
0 = DOWN
1 = FLAT
2 = UP
✓ Encoding confirmed.

BOUNDARY SANITY CHECK
✓ DOWN contains only change <= -30%
✓ FLAT contains only -30% < change < +50%
✓ UP contains only change >= +50%

FINAL TARGET DISTRIBUTION


,target,target_label,count,percentage
0,0,DOWN,206390,32.93
1,1,FLAT,156653,24.99
2,2,UP,263793,42.08



EARLY DROP BASELINE
Early Drop signals : 194,851
Early Drop rate    : 31.08%

Rule:
gsc_impressions_last_30d < gsc_impressions_prev_30d

FINAL ROLLING90WINDOW DATASET
Rows    : 626,836
Columns : 36

All columns:
 1. content_hash_id
 2. client_hash_id
 3. window_start
 4. window_end
 5. future_start
 6. future_end
 7. gsc_clicks_mean_3m
 8. gsc_clicks_last
 9. gsc_impressions_mean_3m
10. gsc_impressions_last
11. gsc_avg_position_mean_3m
12. gsc_avg_position_last
13. ga4_total_engagement_sec_mean_3m
14. ga4_total_engagement_sec_last
15. sessions_organic_mean_3m
16. sessions_organic_last
17. sessions_ai_mean_3m
18. sessions_ai_last
19. gsc_avg_position_missing_mean_3m
20. gsc_avg_position_missing_last
21. ctr_mean_3m
22. ctr_last
23. sec_per_click_mean_3m
24. sec_per_click_last
25. ai_share_mean_3m
26. ai_share_last
27. engagement_per_organic_session_mean_3m
28. engagement_per_organic_session_last
29. current_imp_3m
30. gsc_impressions_prev_30d
31. gsc_impressions_last_30d
32. early_drop

,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,gsc_impressions_prev_30d,gsc_impressions_last_30d,early_drop_signal,future_imp_3m,future_impression_change_pct,target,target_label
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,136.666667,7.0,257.0,False,378.333333,176.829268,2,UP
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,180.666667,146.0,139.0,True,506.666667,180.442804,2,UP
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,216.666667,257.0,254.0,True,487.666667,125.076923,2,UP
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,378.333333,139.0,742.0,False,280.333333,-25.903084,1,FLAT
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,506.666667,254.0,524.0,False,167.000000,-67.039474,0,DOWN
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.666667,0.0,487.666667,197.0,...,0.0,0.0,487.666667,742.0,197.0,True,106.333333,-78.195489,0,DOWN
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.333333,0.0,280.333333,120.0,...,0.0,0.0,280.333333,524.0,120.0,True,74.333333,-73.483948,0,DOWN
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,167.000000,184.0,...,0.0,0.0,167.000000,197.0,184.0,True,41.666667,-75.049900,0,DOWN
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.000000,0.0,106.333333,15.0,...,0.0,0.0,106.333333,120.0,15.0,True,63.666667,-40.125392,0,DOWN
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,2026-02-01,2026-03-01,2026-05-01,0.000000,0.0,74.333333,24.0,...,0.0,0.0,74.333333,184.0,24.0,True,82.666667,11.210762,1,FLAT



FINAL ROLLING90WINDOW SAVED
/content/finalrolling90window.parquet

BLOCK 4.6 COMPLETE
✓ 90-day rolling features retained
✓ Future 90-day target-generation data retained
✓ Previous 30-day impressions retained
✓ Last 30-day impressions retained
✓ Early Drop baseline signal retained
✓ Target created: 0=DOWN, 1=FLAT, 2=UP
✓ Target labels created
✓ Final parquet saved


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method Choice and Why

### Strategy Overview
To address the SEO content decay task, we evaluate a combination of linear benchmarks, non-linear tree ensembles, and validation techniques. We select **Random Forest Classifier** as our primary champion model and **Logistic Regression** as our linear benchmark model.

---

### Comparison of Toolkit Methods

| Toolkit Method | Role in Workflow | Key Strengths | Why Included or Excluded for Decay Lane |
| :--- | :--- | :--- | :--- |
| **Correlation & Signal Analysis** | Feature Screening | Identifies multicollinearity and target leakage. | **Included (Pre-processing):** Essential to strictly remove forbidden trend variables (`trend_pct`, `trend_direction`). |
| **Grouped Validation** | Validation Design | Prevents data leakage across same-client pages. | **Included (Validation):** Grouping by `client_id` ensures the model generalizes to unseen domains rather than memorizing domain-specific baseline numbers. |
| **Logistic Regression** | Baseline ML Model | Simple, fast, and directly interpretable linear baseline. | **Included (ML Baseline):** Benchmark model to verify if a learned linear boundary beats our Week 4 rule-based baseline. |
| **Decision Tree** | Interpretable Model | Visualizable if-else logic trees. | **Included (Secondary):** Useful for quick rules extraction, though prone to higher variance on continuous traffic signals compared to ensembles. |
| **Random Forest** | **Primary Champion Model** | Ensemble of decision trees; handles non-linearities, outliers, and feature interactions. | **SELECTED CHAMPION:** Perfectly fits the power-law nature of web traffic and non-linear ranking drops. |
| **Gradient Boosting** | High-Capacity Model | Strong predictive power on structured tabular data. | **Tested with Constraints:** Evaluated cautiously with shallow depth to avoid overfitting noisy month-to-month traffic fluctuations. |
| **Permutation Importance** | Post-Hoc Interpretability | Measures true feature contribution by shuffling values post-training. | **Included (Sanity Check):** Verifies model honesty and guards against hidden proxy data leakage. |
| **Clustering (K-Means)** | Unsupervised Analysis | Segments content items into distinct performance tiers. | **Exploratory:** Used to analyze structural performance clusters prior to classification. |

---

### Why Random Forest Fits Our SEO Content Decay Lane

1. **Captures Non-Linear SEO Ranking Dynamics:**
   SEO ranking drops do not decay linearly. Losing Rank 1 to Rank 4 results in a catastrophic drop ($\approx 50\%+$) in impressions and CTR, whereas dropping from Rank 25 to Rank 28 has negligible impact. Random Forest handles these step-function thresholds naturally without requiring non-linear feature transformations.

2. **Robust to Heavy-Tailed Power-Law Distributions:**
   Search traffic (`gsc_impressions`) follows a steep power-law distribution where a small percentage of high-traffic pages dominate total volume. Random Forest uses threshold-based splits rather than distance metrics, making it scale-invariant and immune to extreme traffic outliers.

3. **Handles Multi-Signal Feature Interactions:**
   Content decay is rarely caused by a single metric. Random Forest automatically captures multi-variable interaction logic (e.g., *low impressions AND dropping position AND low engagement*) without requiring manual feature engineering.

4. **Transparent Feature Importance & Leakage Defense:**
   Combined with Permutation Importance, Random Forest provides clear insight into which features drive predictions. This ensures the model relies on true signals rather than memorizing forbidden trend indicators.

**Audit to check pages coverage in windows:**

In [18]:
import pandas as pd

# 1. Parquet file load karein
file_path = "/content/finalrolling90window.parquet"
df = pd.read_parquet(file_path)

# 2. Actual dataset columns mapping
page_col = 'content_hash_id'   # Page ID
window_col = 'window_start'    # Window Identifier
client_col = 'client_hash_id'  # Client ID

# 3. Overall Dataset Metrics
total_rows = len(df)
total_unique_pages = df[page_col].nunique()
total_unique_clients = df[client_col].nunique()

# 4. Window-wise Audit Table
audit_df = df.groupby(window_col).agg(
    total_rows=(page_col, 'count'),
    unique_pages=(page_col, 'nunique'),
    unique_clients=(client_col, 'nunique')
).reset_index()

# 5. Percentage Calculations
audit_df['page_coverage_pct'] = ((audit_df['unique_pages'] / total_unique_pages) * 100).round(2)
audit_df['row_share_pct'] = ((audit_df['total_rows'] / total_rows) * 100).round(2)

# Output Print
print("=== OVERALL DATASET METRICS ===")
print(f"Total Rows: {total_rows:,}")
print(f"Total Unique Clients: {total_unique_clients:,}")
print(f"Total Unique Pages (Content Hashes): {total_unique_pages:,}\n")

print("=== WINDOW AUDIT REPORT ===")
print(audit_df.to_string(index=False))

=== OVERALL DATASET METRICS ===
Total Rows: 626,836
Total Unique Clients: 36
Total Unique Pages (Content Hashes): 150,997

=== WINDOW AUDIT REPORT ===
window_start  total_rows  unique_pages  unique_clients  page_coverage_pct  row_share_pct
  2025-01-01         173           173               2               0.11           0.03
  2025-02-01        4388          4388               3               2.91           0.70
  2025-03-01        8723          8723               4               5.78           1.39
  2025-04-01       10879         10879               4               7.20           1.74
  2025-05-01       11860         11860               4               7.85           1.89
  2025-06-01       13482         13482               9               8.93           2.15
  2025-07-01       23387         23387              14              15.49           3.73
  2025-08-01       32357         32357              15              21.43           5.16
  2025-09-01       50214         50214          

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
import pandas as pd
import numpy as np

# 1. Dataset load karein
file_path = "/content/finalrolling90window.parquet"
df = pd.read_parquet(file_path)
df['window_start'] = pd.to_datetime(df['window_start'])

total_rows = len(df)
total_pages = df['content_hash_id'].nunique()
total_clients = df['client_hash_id'].nunique()

print("==================================================")
print("=== APPROACH 1: LATE TEMPORAL SPLIT (2026-01-01) ==")
print("==================================================")

# Threshold at 2026-01-01 (2025 full = Train, Jan 2026 = Test)
temp_threshold = pd.to_datetime('2026-01-01')
train_temp = df[df['window_start'] < temp_threshold]
test_temp = df[df['window_start'] >= temp_threshold]

print(f"TRAIN: {len(train_temp):,} rows ({len(train_temp)/total_rows*100:.2f}%) | {train_temp['content_hash_id'].nunique():,} pages | {train_temp['client_hash_id'].nunique()} clients")
print(f"TEST : {len(test_temp):,} rows ({len(test_temp)/total_rows*100:.2f}%) | {test_temp['content_hash_id'].nunique():,} pages | {test_temp['client_hash_id'].nunique()} clients\n")


print("==================================================")
print("=== APPROACH 2: GROUPED BY CLIENT SPLIT (80/20) ===")
print("==================================================")

# Client basis par 80-20 split (Unseen websites test karne ke liye)
np.random.seed(42)
unique_clients = df['client_hash_id'].unique()
np.random.shuffle(unique_clients)

train_client_count = int(len(unique_clients) * 0.8)
train_clients = unique_clients[:train_client_count]
test_clients = unique_clients[train_client_count:]

train_grp = df[df['client_hash_id'].isin(train_clients)]
test_grp = df[df['client_hash_id'].isin(test_clients)]

print(f"TRAIN: {len(train_grp):,} rows ({len(train_grp)/total_rows*100:.2f}%) | {train_grp['content_hash_id'].nunique():,} pages | {len(train_clients)} clients")
print(f"TEST : {len(test_grp):,} rows ({len(test_grp)/total_rows*100:.2f}%) | {test_grp['content_hash_id'].nunique():,} pages | {len(test_clients)} clients")

=== APPROACH 1: LATE TEMPORAL SPLIT (2026-01-01) ==
TRAIN: 486,894 rows (77.67%) | 136,266 pages | 36 clients
TEST : 139,942 rows (22.33%) | 139,942 pages | 32 clients

=== APPROACH 2: GROUPED BY CLIENT SPLIT (80/20) ===
TRAIN: 571,381 rows (91.15%) | 134,143 pages | 28 clients
TEST : 55,455 rows (8.85%) | 16,854 pages | 8 clients


**Proper leakage audit on this split**

In [21]:
# ============================================================
# FINAL SPLIT + LEAKAGE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("FINAL TEMPORAL SPLIT + LEAKAGE AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD FINAL DATASET
# ------------------------------------------------------------

file_path = "/content/finalrolling90window.parquet"

df = pd.read_parquet(file_path)

df["window_start"] = pd.to_datetime(
    df["window_start"],
    errors="coerce"
)

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

# ------------------------------------------------------------
# 2. TARGET VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET VERIFICATION")
print("=" * 80)

assert "target" in df.columns
assert "target_label" in df.columns

print("Target unique values:", sorted(df["target"].dropna().unique()))

assert set(df["target"].dropna().unique()).issubset({0, 1, 2})

print("✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")

# ------------------------------------------------------------
# 3. FUTURE COLUMN AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FUTURE COLUMN AUDIT")
print("=" * 80)

future_columns = [
    c for c in df.columns
    if c.lower().startswith("future_")
]

print("Future columns found:")

for col in future_columns:
    print(" -", col)

print(f"\nTotal future columns: {len(future_columns)}")

# These are allowed to exist in final dataset.
# They MUST NOT enter X.

# ------------------------------------------------------------
# 4. EXPLICIT TARGET / LEAKAGE COLUMNS
# ------------------------------------------------------------

target_columns = {
    "target",
    "target_label",
    "future_impression_change_pct",
    "future_imp_3m"
}

leakage_columns = (
    target_columns
    | set(future_columns)
)

print("\n" + "=" * 80)
print("TARGET / FUTURE LEAKAGE COLUMNS")
print("=" * 80)

for col in sorted(leakage_columns):
    if col in df.columns:
        print("BLOCKED:", col)

# ------------------------------------------------------------
# 5. TEMPORAL SPLIT
# ------------------------------------------------------------

cutoff = pd.Timestamp("2026-01-01")

train = df[
    df["window_start"] < cutoff
].copy()

test = df[
    df["window_start"] >= cutoff
].copy()

print("\n" + "=" * 80)
print("TEMPORAL SPLIT")
print("=" * 80)

print(f"Cutoff : {cutoff.date()}")

print(
    f"TRAIN rows : {len(train):,} "
    f"({len(train)/len(df)*100:.2f}%)"
)

print(
    f"TEST rows  : {len(test):,} "
    f"({len(test)/len(df)*100:.2f}%)"
)

# ------------------------------------------------------------
# 6. TEMPORAL SAFETY
# ------------------------------------------------------------

train_max = train["window_start"].max()
test_min = test["window_start"].min()

print("\nTrain latest window :", train_max.date())
print("Test earliest window:", test_min.date())

assert train_max < cutoff
assert test_min >= cutoff

print("✓ Temporal ordering is correct.")

# ------------------------------------------------------------
# 7. PAGE OVERLAP
# ------------------------------------------------------------

train_pages = set(
    train["content_hash_id"].dropna().unique()
)

test_pages = set(
    test["content_hash_id"].dropna().unique()
)

page_overlap = (
    train_pages &
    test_pages
)

print("\n" + "=" * 80)
print("PAGE OVERLAP AUDIT")
print("=" * 80)

print(f"Train pages : {len(train_pages):,}")
print(f"Test pages  : {len(test_pages):,}")
print(f"Overlap     : {len(page_overlap):,}")

if len(page_overlap) > 0:
    print(
        "\n✓ Page overlap exists — this is expected "
        "for future prediction of existing pages."
    )
else:
    print("✓ No page overlap.")

# ------------------------------------------------------------
# 8. CLIENT OVERLAP
# ------------------------------------------------------------

train_clients = set(
    train["client_hash_id"].dropna().unique()
)

test_clients = set(
    test["client_hash_id"].dropna().unique()
)

client_overlap = (
    train_clients &
    test_clients
)

print("\n" + "=" * 80)
print("CLIENT OVERLAP AUDIT")
print("=" * 80)

print(f"Train clients : {len(train_clients):,}")
print(f"Test clients  : {len(test_clients):,}")
print(f"Overlap       : {len(client_overlap):,}")

# Client overlap is NOT leakage for the primary
# existing-client future prediction objective.

# ------------------------------------------------------------
# 9. MODEL FEATURE CANDIDATES
# ------------------------------------------------------------

excluded = {
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",
    "month",
    "target",
    "target_label"
}

excluded.update(future_columns)

candidate_features = [
    c for c in df.columns
    if c not in excluded
]

# ------------------------------------------------------------
# 10. SECONDARY NAME-BASED LEAKAGE CHECK
# ------------------------------------------------------------

suspicious_features = []

for col in candidate_features:

    name = col.lower()

    suspicious_words = [
        "future",
        "target",
        "label",
        "next_3m",
        "next_90d",
        "decay_rate"
    ]

    if any(word in name for word in suspicious_words):
        suspicious_features.append(col)

print("\n" + "=" * 80)
print("SUSPICIOUS FEATURE-NAME AUDIT")
print("=" * 80)

if suspicious_features:

    for col in suspicious_features:
        print("REVIEW:", col)

else:
    print("✓ No suspicious feature names found.")

# ------------------------------------------------------------
# 11. BUILD X / y
# ------------------------------------------------------------

X_train = train[candidate_features].copy()
X_test = test[candidate_features].copy()

y_train = train["target"].copy()
y_test = test["target"].copy()

print("\n" + "=" * 80)
print("MODEL INPUT")
print("=" * 80)

print("Number of features:", len(candidate_features))
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# ------------------------------------------------------------
# 12. FINAL NA CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING VALUE CHECK")
print("=" * 80)

train_missing = X_train.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

print("Train missing values:", train_missing)
print("Test missing values :", test_missing)

# ------------------------------------------------------------
# 13. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 80)

train_dist = (
    y_train
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

train_dist["percentage"] = (
    train_dist["count"]
    / len(y_train)
    * 100
).round(2)

display(train_dist)

print("\n" + "=" * 80)
print("TEST TARGET DISTRIBUTION")
print("=" * 80)

test_dist = (
    y_test
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

test_dist["percentage"] = (
    test_dist["count"]
    / len(y_test)
    * 100
).round(2)

display(test_dist)

# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL AUDIT VERDICT")
print("=" * 80)

print("✓ Temporal cutoff applied.")
print("✓ Test contains later windows than training.")
print("✓ Future columns excluded from X.")
print("✓ Target columns excluded from X.")
print("✓ Target encoding verified.")
print("✓ Existing-page overlap is allowed for future prediction.")
print("✓ No random row mixing.")
print("✓ No artificial class balancing.")
print("✓ Candidate features detected from final dataset.")

print("\nFINAL FEATURE COUNT:", len(candidate_features))

FINAL TEMPORAL SPLIT + LEAKAGE AUDIT
Rows    : 626,836
Columns : 36

TARGET VERIFICATION
Target unique values: [np.int8(0), np.int8(1), np.int8(2)]
✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP

FUTURE COLUMN AUDIT
Future columns found:
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct

Total future columns: 4

TARGET / FUTURE LEAKAGE COLUMNS
BLOCKED: future_end
BLOCKED: future_imp_3m
BLOCKED: future_impression_change_pct
BLOCKED: future_start
BLOCKED: target
BLOCKED: target_label

TEMPORAL SPLIT
Cutoff : 2026-01-01
TRAIN rows : 486,894 (77.67%)
TEST rows  : 139,942 (22.33%)

Train latest window : 2025-12-01
Test earliest window: 2026-01-01
✓ Temporal ordering is correct.

PAGE OVERLAP AUDIT
Train pages : 136,266
Test pages  : 139,942
Overlap     : 125,211

✓ Page overlap exists — this is expected for future prediction of existing pages.

CLIENT OVERLAP AUDIT
Train clients : 36
Test clients  : 32
Overlap       : 32

SUSPICIOUS FEATURE-NAME AUDIT
✓ No suspicious feature na

,target,count,percentage
0,0,132136,27.14
1,1,119430,24.53
2,2,235328,48.33



TEST TARGET DISTRIBUTION


,target,count,percentage
0,0,74254,53.06
1,1,37223,26.60
2,2,28465,20.34



FINAL AUDIT VERDICT
✓ Temporal cutoff applied.
✓ Test contains later windows than training.
✓ Future columns excluded from X.
✓ Target columns excluded from X.
✓ Target encoding verified.
✓ Existing-page overlap is allowed for future prediction.
✓ No random row mixing.
✓ No artificial class balancing.
✓ Candidate features detected from final dataset.

FINAL FEATURE COUNT: 26


Selected Evaluation Strategy
Late Temporal Split (Cutoff Date: 2026-01-01)

Train Set (2025-01-01 to 2025-12-01): 486,894 rows (77.67%) | 136,266 unique pages | 36 clients

Test Set (2026-01-01): 139,942 rows (22.33%) | 139,942 unique pages | 32 clients

💡 Justification & Reasons
Zero Data Leakage: SEO traffic forecasting time-series problem hai. Strictly 2026-01-01 par cut karne se future window data train set me leak hone se bach jata hai.

Production-Like Simulation: Real-world deployment ko simulate karta hai jahan historical trends se future performance forecast ki jati hai.

Ideal Split Ratio (~78/22): Without artificial downsampling, naturally balanced 77.67% Train aur 22.33% Test distribution milti hai.

Feature Generalization: Model static page IDs memorized karne ke bajaye real GSC impressions aur engagement dynamics ke actual signals seekhta hai.

**Block 5.1 — Final Rolling Dataset Load + Structure Check**

In [3]:
# ============================================================
# BLOCK 5.1 — FINAL ROLLING DATASET LOAD + STRUCTURE CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 5.1 — FINAL ROLLING DATASET LOAD + STRUCTURE CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD FINAL DATASET
# ------------------------------------------------------------

file_path = "/content/finalrolling90window.parquet"

df_final = pd.read_parquet(file_path)

# ------------------------------------------------------------
# 2. BASIC INFORMATION
# ------------------------------------------------------------

print(f"\nSource file : {file_path}")
print(f"Rows        : {len(df_final):,}")
print(f"Columns     : {df_final.shape[1]}")

# ------------------------------------------------------------
# 3. REQUIRED CORE COLUMNS
# ------------------------------------------------------------

required_columns = [
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",
    "target",
    "target_label"
]

missing_required = [
    col for col in required_columns
    if col not in df_final.columns
]

if missing_required:
    raise KeyError(
        "Required columns missing:\n"
        + "\n".join(missing_required)
    )

# ------------------------------------------------------------
# 4. DATE CONVERSION
# ------------------------------------------------------------

df_final["window_start"] = pd.to_datetime(
    df_final["window_start"],
    errors="coerce"
)

df_final["window_end"] = pd.to_datetime(
    df_final["window_end"],
    errors="coerce"
)

if df_final["window_start"].isna().any():
    raise ValueError(
        "window_start contains invalid/missing dates."
    )

# ------------------------------------------------------------
# 5. TARGET VALIDATION
# ------------------------------------------------------------

target_values = set(
    pd.to_numeric(
        df_final["target"],
        errors="coerce"
    ).dropna().unique()
)

print("\n" + "=" * 80)
print("TARGET VALIDATION")
print("=" * 80)

print("Target values:", sorted(target_values))

if not target_values.issubset({0, 1, 2}):
    raise ValueError(
        f"Unexpected target values: {target_values}"
    )

if df_final["target"].isna().any():
    raise ValueError("Target contains missing values.")

print("✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")

# ------------------------------------------------------------
# 6. TARGET LABEL VALIDATION
# ------------------------------------------------------------

expected_labels = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

label_check = (
    df_final[["target", "target_label"]]
    .drop_duplicates()
    .sort_values("target")
)

print("\nTarget mapping:")
display(label_check)

for target_value, label in expected_labels.items():

    rows = df_final[
        df_final["target"] == target_value
    ]

    if len(rows) > 0:

        actual_labels = set(
            rows["target_label"]
            .astype(str)
            .str.upper()
            .unique()
        )

        if actual_labels != {label}:
            raise ValueError(
                f"Target mapping incorrect for "
                f"{target_value}: {actual_labels}"
            )

print("✓ Target encoding confirmed.")

# ------------------------------------------------------------
# 7. FUTURE COLUMN AUDIT
# ------------------------------------------------------------

future_columns = [
    col for col in df_final.columns
    if col.startswith("future_")
]

print("\n" + "=" * 80)
print("FUTURE COLUMN AUDIT")
print("=" * 80)

for col in future_columns:
    print(" -", col)

print(f"\nFuture columns found: {len(future_columns)}")

# ------------------------------------------------------------
# 8. FINAL STRUCTURE PREVIEW
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL DATASET PREVIEW")
print("=" * 80)

display(df_final.head(5))

print("\n✓ BLOCK 5.1 COMPLETE")

BLOCK 5.1 — FINAL ROLLING DATASET LOAD + STRUCTURE CHECK

Source file : /content/finalrolling90window.parquet
Rows        : 626,836
Columns     : 36

TARGET VALIDATION
Target values: [np.int8(0), np.int8(1), np.int8(2)]
✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP

Target mapping:


,target,target_label
4,0,DOWN
3,1,FLAT
0,2,UP


✓ Target encoding confirmed.

FUTURE COLUMN AUDIT
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct

Future columns found: 4

FINAL DATASET PREVIEW


,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,gsc_impressions_prev_30d,gsc_impressions_last_30d,early_drop_signal,future_imp_3m,future_impression_change_pct,target,target_label
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,136.666667,7.0,257.0,False,378.333333,176.829268,2,UP
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,180.666667,146.0,139.0,True,506.666667,180.442804,2,UP
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,216.666667,257.0,254.0,True,487.666667,125.076923,2,UP
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,378.333333,139.0,742.0,False,280.333333,-25.903084,1,FLAT
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,506.666667,254.0,524.0,False,167.000000,-67.039474,0,DOWN



✓ BLOCK 5.1 COMPLETE


**BLOCK 5.2 — SEPARATE X/y + LEAKAGE CHECK**

In [4]:
# ============================================================
# BLOCK 5.2 — SEPARATE X / y + LEAKAGE CHECK
# ============================================================

print("=" * 80)
print("BLOCK 5.2 — X / y SEPARATION + LEAKAGE AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. COLUMNS THAT MUST NEVER ENTER X
# ------------------------------------------------------------

blocked_columns = {
    # Metadata
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",

    # Target
    "target",
    "target_label",

    # Explicit future information
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct"
}

# ------------------------------------------------------------
# 2. FIND POTENTIAL FUTURE COLUMNS
# ------------------------------------------------------------

future_name_columns = [
    col
    for col in df_final.columns
    if col.lower().startswith("future_")
]

# Add every future_* column to blocked set
blocked_columns.update(future_name_columns)

# ------------------------------------------------------------
# 3. CREATE MODEL FEATURE LIST
# ------------------------------------------------------------

feature_columns = [
    col
    for col in df_final.columns
    if col not in blocked_columns
]

# ------------------------------------------------------------
# 4. SUSPICIOUS NAME AUDIT
# ------------------------------------------------------------

suspicious_keywords = [
    "future",
    "target",
    "label",
    "impression_change",
    "next_",
    "outcome"
]

suspicious_features = [
    col
    for col in feature_columns
    if any(
        keyword in col.lower()
        for keyword in suspicious_keywords
    )
]

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print(f"Candidate model features: {len(feature_columns)}")

if suspicious_features:
    print("\nWARNING — suspicious feature names:")
    for col in suspicious_features:
        print(" -", col)

    raise ValueError(
        "Potential leakage detected in model features."
    )

print("✓ No suspicious feature names found.")

# ------------------------------------------------------------
# 5. VERIFY NO FUTURE / TARGET COLUMNS IN X
# ------------------------------------------------------------

invalid_x_columns = [
    col
    for col in feature_columns
    if (
        col.startswith("future_")
        or col in {"target", "target_label"}
    )
]

if invalid_x_columns:
    raise ValueError(
        "Leakage columns found in X:\n"
        + "\n".join(invalid_x_columns)
    )

# ------------------------------------------------------------
# 6. CREATE X AND y
# ------------------------------------------------------------

X = df_final[feature_columns].copy()

y = pd.to_numeric(
    df_final["target"],
    errors="coerce"
).astype("int8")

# ------------------------------------------------------------
# 7. NUMERIC FEATURE CHECK
# ------------------------------------------------------------

non_numeric_features = [
    col
    for col in X.columns
    if not pd.api.types.is_numeric_dtype(X[col])
]

if non_numeric_features:
    raise TypeError(
        "Non-numeric model features found:\n"
        + "\n".join(non_numeric_features)
    )

# ------------------------------------------------------------
# 8. MISSING VALUE CHECK
# ------------------------------------------------------------

train_ready_missing = X.isna().sum()

missing_features = (
    train_ready_missing[
        train_ready_missing > 0
    ]
)

if len(missing_features) > 0:

    print("\nMissing values found:")
    print(missing_features)

    raise ValueError(
        "Model features contain missing values."
    )

# ------------------------------------------------------------
# 9. TARGET CHECK
# ------------------------------------------------------------

if not set(y.unique()).issubset({0, 1, 2}):
    raise ValueError(
        "Target contains values outside 0,1,2."
    )

# ------------------------------------------------------------
# 10. FINAL FEATURE LIST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL INPUT FEATURES")
print("=" * 80)

for i, col in enumerate(feature_columns, 1):
    print(f"{i:2}. {col}")

print("\n" + "=" * 80)
print("MODEL INPUT SHAPE")
print("=" * 80)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

print("\n✓ No future columns in X")
print("✓ No target columns in X")
print("✓ No target_label in X")
print("✓ No metadata in X")
print("✓ No suspicious feature names")
print("✓ No missing feature values")
print("✓ Target = 0 / 1 / 2")

print("\n✓ BLOCK 5.2 COMPLETE")

BLOCK 5.2 — X / y SEPARATION + LEAKAGE AUDIT

LEAKAGE CHECK
Candidate model features: 26
✓ No suspicious feature names found.

FINAL MODEL INPUT FEATURES
 1. gsc_clicks_mean_3m
 2. gsc_clicks_last
 3. gsc_impressions_mean_3m
 4. gsc_impressions_last
 5. gsc_avg_position_mean_3m
 6. gsc_avg_position_last
 7. ga4_total_engagement_sec_mean_3m
 8. ga4_total_engagement_sec_last
 9. sessions_organic_mean_3m
10. sessions_organic_last
11. sessions_ai_mean_3m
12. sessions_ai_last
13. gsc_avg_position_missing_mean_3m
14. gsc_avg_position_missing_last
15. ctr_mean_3m
16. ctr_last
17. sec_per_click_mean_3m
18. sec_per_click_last
19. ai_share_mean_3m
20. ai_share_last
21. engagement_per_organic_session_mean_3m
22. engagement_per_organic_session_last
23. current_imp_3m
24. gsc_impressions_prev_30d
25. gsc_impressions_last_30d
26. early_drop_signal

MODEL INPUT SHAPE
X shape : (626836, 26)
y shape : (626836,)

✓ No future columns in X
✓ No target columns in X
✓ No target_label in X
✓ No metadata in X

**BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT**

In [5]:

# ============================================================
# BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT
# ============================================================

print("=" * 80)
print("BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. TEMPORAL CUTOFF
# ------------------------------------------------------------

temporal_cutoff = pd.Timestamp("2026-01-01")

# ------------------------------------------------------------
# 2. CREATE MASKS
# ------------------------------------------------------------

train_mask = (
    df_final["window_start"] < temporal_cutoff
)

test_mask = (
    df_final["window_start"] >= temporal_cutoff
)

# ------------------------------------------------------------
# 3. SPLIT X / y
# ------------------------------------------------------------

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

# ------------------------------------------------------------
# 4. BASIC SIZE CHECK
# ------------------------------------------------------------

if len(X_train) == 0:
    raise ValueError("Training set is empty.")

if len(X_test) == 0:
    raise ValueError("Test set is empty.")

# ------------------------------------------------------------
# 5. TEMPORAL ORDER CHECK
# ------------------------------------------------------------

train_latest = df_final.loc[
    train_mask,
    "window_start"
].max()

test_earliest = df_final.loc[
    test_mask,
    "window_start"
].min()

if train_latest >= test_earliest:
    raise ValueError(
        "Temporal leakage: training extends into test period."
    )

# ------------------------------------------------------------
# 6. PAGE OVERLAP AUDIT
# ------------------------------------------------------------

train_pages = set(
    df_final.loc[
        train_mask,
        "content_hash_id"
    ]
)

test_pages = set(
    df_final.loc[
        test_mask,
        "content_hash_id"
    ]
)

page_overlap = (
    len(train_pages.intersection(test_pages))
)

# Existing-page overlap is allowed because this is
# future trajectory prediction.

# ------------------------------------------------------------
# 7. CLIENT OVERLAP AUDIT
# ------------------------------------------------------------

train_clients = set(
    df_final.loc[
        train_mask,
        "client_hash_id"
    ]
)

test_clients = set(
    df_final.loc[
        test_mask,
        "client_hash_id"
    ]
)

client_overlap = (
    len(train_clients.intersection(test_clients))
)

# ------------------------------------------------------------
# 8. DISTRIBUTION FUNCTION
# ------------------------------------------------------------

def distribution(series):

    result = (
        series
        .value_counts()
        .sort_index()
        .rename_axis("target")
        .reset_index(name="count")
    )

    result["percentage"] = (
        result["count"]
        / len(series)
        * 100
    ).round(2)

    result["label"] = result["target"].map({
        0: "DOWN",
        1: "FLAT",
        2: "UP"
    })

    return result[
        ["target", "label", "count", "percentage"]
    ]

# ------------------------------------------------------------
# 9. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEMPORAL SPLIT")
print("=" * 80)

print(f"Cutoff : {temporal_cutoff.date()}")

print(
    f"TRAIN : {len(X_train):,} "
    f"({len(X_train)/len(X)*100:.2f}%)"
)

print(
    f"TEST  : {len(X_test):,} "
    f"({len(X_test)/len(X)*100:.2f}%)"
)

print(f"\nTrain latest window : {train_latest.date()}")
print(f"Test earliest window: {test_earliest.date()}")

print("\n✓ Temporal ordering verified.")

# ------------------------------------------------------------
# 10. PAGE AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PAGE OVERLAP AUDIT")
print("=" * 80)

print(f"Train pages : {len(train_pages):,}")
print(f"Test pages  : {len(test_pages):,}")
print(f"Overlap     : {page_overlap:,}")

print(
    "\n✓ Existing-page overlap is allowed "
    "for future trajectory prediction."
)

# ------------------------------------------------------------
# 11. CLIENT AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CLIENT OVERLAP AUDIT")
print("=" * 80)

print(f"Train clients : {len(train_clients):,}")
print(f"Test clients  : {len(test_clients):,}")
print(f"Overlap       : {client_overlap:,}")

print(
    "\nNote: Client overlap is expected because "
    "the model predicts future behavior of existing clients/pages."
)

# ------------------------------------------------------------
# 12. TRAIN DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 80)

display(
    distribution(y_train)
)

# ------------------------------------------------------------
# 13. TEST DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST TARGET DISTRIBUTION")
print("=" * 80)

display(
    distribution(y_test)
)

# ------------------------------------------------------------
# 14. FINAL SHAPES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SPLIT SHAPES")
print("=" * 80)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

# ------------------------------------------------------------
# 15. FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert train_latest < test_earliest

assert not any(
    col.startswith("future_")
    for col in X_train.columns
)

assert "target" not in X_train.columns
assert "target_label" not in X_train.columns

print("\n✓ No temporal leakage.")
print("✓ No future columns in training/testing.")
print("✓ No target columns in training/testing.")
print("✓ Split is strictly time-aware.")

print("\n✓ BLOCK 5.3 COMPLETE")

BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT

TEMPORAL SPLIT
Cutoff : 2026-01-01
TRAIN : 486,894 (77.67%)
TEST  : 139,942 (22.33%)

Train latest window : 2025-12-01
Test earliest window: 2026-01-01

✓ Temporal ordering verified.

PAGE OVERLAP AUDIT
Train pages : 136,266
Test pages  : 139,942
Overlap     : 125,211

✓ Existing-page overlap is allowed for future trajectory prediction.

CLIENT OVERLAP AUDIT
Train clients : 36
Test clients  : 32
Overlap       : 32

Note: Client overlap is expected because the model predicts future behavior of existing clients/pages.

TRAIN TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,132136,27.14
1,1,FLAT,119430,24.53
2,2,UP,235328,48.33



TEST TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,74254,53.06
1,1,FLAT,37223,26.60
2,2,UP,28465,20.34



FINAL SPLIT SHAPES
X_train : (486894, 26)
X_test  : (139942, 26)
y_train : (486894,)
y_test  : (139942,)

✓ No temporal leakage.
✓ No future columns in training/testing.
✓ No target columns in training/testing.
✓ Split is strictly time-aware.

✓ BLOCK 5.3 COMPLETE


**BLOCK A — Target Quality Audit**

In [9]:
# ================================================================
# AUDIT A — TARGET QUALITY + THRESHOLD SANITY CHECK
# ================================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("AUDIT A — TARGET QUALITY + THRESHOLD SANITY CHECK")
print("=" * 80)

# ------------------------------------------------
# 1. LOAD FINAL DATASET
# ------------------------------------------------

path = "/content/finalrolling90window.parquet"

df_audit = pd.read_parquet(path)

df_audit["window_start"] = pd.to_datetime(
    df_audit["window_start"],
    errors="coerce"
)

df_audit["future_start"] = pd.to_datetime(
    df_audit["future_start"],
    errors="coerce"
)

print(f"\nRows : {len(df_audit):,}")
print(f"Cols : {df_audit.shape[1]}")

# ------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------

required = [
    "window_start",
    "future_start",
    "future_impression_change_pct",
    "target"
]

missing = [
    c for c in required
    if c not in df_audit.columns
]

if missing:
    raise KeyError(
        "Missing required columns:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------
# 3. RE-CALCULATE TARGET INDEPENDENTLY
# ------------------------------------------------

change = pd.to_numeric(
    df_audit["future_impression_change_pct"],
    errors="coerce"
)

recalculated_target = np.select(
    [
        change <= -30,
        change < 50
    ],
    [
        0,
        1
    ],
    default=2
).astype("int8")

stored_target = pd.to_numeric(
    df_audit["target"],
    errors="coerce"
)

# ------------------------------------------------
# 4. TARGET CONSISTENCY
# ------------------------------------------------

mismatch = (
    stored_target != recalculated_target
)

print("\n" + "=" * 80)
print("TARGET CONSISTENCY")
print("=" * 80)

print(
    f"Target mismatches : {mismatch.sum():,}"
)

assert mismatch.sum() == 0, (
    "ERROR: Stored target does not match "
    "the frozen -30 / +50 rule."
)

print("✓ Target exactly matches frozen rule.")

# ------------------------------------------------
# 5. CLASS DISTRIBUTION
# ------------------------------------------------

labels = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

summary = (
    stored_target
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

summary["label"] = summary["target"].map(labels)

summary["percentage"] = (
    summary["count"] /
    len(df_audit) * 100
).round(2)

print("\n" + "=" * 80)
print("OVERALL TARGET DISTRIBUTION")
print("=" * 80)

display(
    summary[
        ["target", "label", "count", "percentage"]
    ]
)

# ------------------------------------------------
# 6. BOUNDARY TEST
# ------------------------------------------------

print("\n" + "=" * 80)
print("BOUNDARY SANITY CHECK")
print("=" * 80)

boundary_checks = {
    "DOWN <= -30": (
        (stored_target == 0)
        & (change > -30)
    ).sum(),

    "FLAT > -30": (
        (stored_target == 1)
        & (change <= -30)
    ).sum(),

    "FLAT < +50": (
        (stored_target == 1)
        & (change >= 50)
    ).sum(),

    "UP >= +50": (
        (stored_target == 2)
        & (change < 50)
    ).sum()
}

for name, count in boundary_checks.items():
    print(f"{name:<20}: {count:,}")

assert all(
    count == 0
    for count in boundary_checks.values()
)

print("\n✓ All target boundaries are correct.")

# ------------------------------------------------
# 7. CHANGE DISTRIBUTION
# ------------------------------------------------

print("\n" + "=" * 80)
print("CHANGE DISTRIBUTION")
print("=" * 80)

stats = pd.Series({
    "Minimum": change.min(),
    "Q01": change.quantile(.01),
    "Q05": change.quantile(.05),
    "Q10": change.quantile(.10),
    "Q25": change.quantile(.25),
    "Median": change.median(),
    "Q75": change.quantile(.75),
    "Q90": change.quantile(.90),
    "Q95": change.quantile(.95),
    "Q99": change.quantile(.99),
    "Maximum": change.max()
})

display(
    stats.round(2).rename("change_pct").to_frame()
)

# ------------------------------------------------
# 8. EXTREME VALUES
# ------------------------------------------------

print("\n" + "=" * 80)
print("EXTREME CHANGE CHECK")
print("=" * 80)

print(
    "Rows <= -90% :",
    (change <= -90).sum()
)

print(
    "Rows >= +500%:",
    (change >= 500).sum()
)

print(
    "Rows >= +1000%:",
    (change >= 1000).sum()
)

# ------------------------------------------------
# 9. CURRENT VS FUTURE IMPRESSIONS
# ------------------------------------------------

for col in [
    "current_imp_3m",
    "future_imp_3m"
]:
    if col in df_audit.columns:
        df_audit[col] = pd.to_numeric(
            df_audit[col],
            errors="coerce"
        )

print("\n" + "=" * 80)
print("ZERO / VERY LOW IMPRESSION CHECK")
print("=" * 80)

if "current_imp_3m" in df_audit.columns:

    print(
        "Current 3M impressions = 0:",
        (df_audit["current_imp_3m"] == 0).sum()
    )

    print(
        "Current 3M impressions < 10:",
        (df_audit["current_imp_3m"] < 10).sum()
    )

if "future_imp_3m" in df_audit.columns:

    print(
        "Future 3M impressions = 0:",
        (df_audit["future_imp_3m"] == 0).sum()
    )

print("\n" + "=" * 80)
print("AUDIT A COMPLETE")
print("=" * 80)

AUDIT A — TARGET QUALITY + THRESHOLD SANITY CHECK

Rows : 626,836
Cols : 36

TARGET CONSISTENCY
Target mismatches : 0
✓ Target exactly matches frozen rule.

OVERALL TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,206390,32.93
1,1,FLAT,156653,24.99
2,2,UP,263793,42.08



BOUNDARY SANITY CHECK
DOWN <= -30         : 0
FLAT > -30          : 0
FLAT < +50          : 0
UP >= +50           : 0

✓ All target boundaries are correct.

CHANGE DISTRIBUTION


,change_pct
Minimum,-100.00
Q01,-100.00
Q05,-100.00
Q10,-90.00
Q25,-50.16
Median,19.57
Q75,150.00
Q90,442.13
Q95,909.09
Q99,4678.11



EXTREME CHANGE CHECK
Rows <= -90% : 62514
Rows >= +500%: 56140
Rows >= +1000%: 28828

ZERO / VERY LOW IMPRESSION CHECK
Current 3M impressions = 0: 0
Current 3M impressions < 10: 152759
Future 3M impressions = 0: 45688

AUDIT A COMPLETE


**BLOCK B — Temporal Target Drift Audit**

In [10]:
# ================================================================
# AUDIT B — TEMPORAL TARGET DRIFT
# ================================================================

print("=" * 80)
print("AUDIT B — TEMPORAL TARGET DRIFT")
print("=" * 80)

df_b = df_audit.copy()

df_b["year_month"] = (
    df_b["window_start"]
    .dt.to_period("M")
)

# ------------------------------------------------
# MONTHLY TARGET COUNTS
# ------------------------------------------------

monthly_counts = pd.crosstab(
    df_b["year_month"],
    df_b["target"]
)

for cls in [0, 1, 2]:
    if cls not in monthly_counts.columns:
        monthly_counts[cls] = 0

monthly_counts = monthly_counts[
    [0, 1, 2]
]

monthly_pct = (
    monthly_counts
    .div(monthly_counts.sum(axis=1), axis=0)
    * 100
).round(2)

monthly_pct.columns = [
    "DOWN_%",
    "FLAT_%",
    "UP_%"
]

monthly_pct = monthly_pct.reset_index()

print("\n" + "=" * 80)
print("MONTHLY TARGET DISTRIBUTION")
print("=" * 80)

display(monthly_pct)

# ------------------------------------------------
# EARLY VS LATE PERIOD
# ------------------------------------------------

cutoff = pd.Timestamp("2026-01-01")

early = df_b[
    df_b["window_start"] < cutoff
]

late = df_b[
    df_b["window_start"] >= cutoff
]

def distribution(data):

    counts = (
        data["target"]
        .value_counts()
        .reindex([0, 1, 2], fill_value=0)
    )

    return pd.DataFrame({
        "count": counts,
        "percentage": (
            counts / len(data) * 100
        ).round(2)
    }, index=["DOWN", "FLAT", "UP"])

print("\n" + "=" * 80)
print("BEFORE 2026-01-01")
print("=" * 80)

display(distribution(early))

print("\n" + "=" * 80)
print("FROM 2026-01-01")
print("=" * 80)

display(distribution(late))

# ------------------------------------------------
# DRIFT DIFFERENCE
# ------------------------------------------------

early_dist = (
    early["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    * 100
)

late_dist = (
    late["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    * 100
)

drift = pd.DataFrame({
    "early_pct": early_dist.round(2),
    "late_pct": late_dist.round(2),
    "change_pp": (
        late_dist - early_dist
    ).round(2)
})

drift.index = [
    "DOWN",
    "FLAT",
    "UP"
]

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION DRIFT")
print("=" * 80)

display(drift)

print("\n" + "=" * 80)
print("AUDIT B COMPLETE")
print("=" * 80)

AUDIT B — TEMPORAL TARGET DRIFT

MONTHLY TARGET DISTRIBUTION


,year_month,DOWN_%,FLAT_%,UP_%
0,2025-01,3.47,10.40,86.13
1,2025-02,16.52,22.95,60.53
2,2025-03,21.39,30.80,47.80
3,2025-04,27.95,38.44,33.61
4,2025-05,33.54,39.31,27.15
5,2025-06,40.51,34.61,24.88
6,2025-07,31.36,28.10,40.55
7,2025-08,28.58,24.99,46.42
8,2025-09,21.50,20.74,57.77
9,2025-10,19.77,15.14,65.08



BEFORE 2026-01-01


,count,percentage
DOWN,NaN,NaN
FLAT,NaN,NaN
UP,NaN,NaN



FROM 2026-01-01


,count,percentage
DOWN,NaN,NaN
FLAT,NaN,NaN
UP,NaN,NaN



TARGET DISTRIBUTION DRIFT


,early_pct,late_pct,change_pp
DOWN,27.14,53.06,25.92
FLAT,24.53,26.60,2.07
UP,48.33,20.34,-27.99



AUDIT B COMPLETE


**BLOCK C — Feature Signal Audit**

In [11]:
# ================================================================
# AUDIT C — FEATURE → TARGET SIGNAL
# ================================================================

print("=" * 80)
print("AUDIT C — FEATURE PREDICTIVE SIGNAL")
print("=" * 80)

df_c = df_audit.copy()

# ------------------------------------------------
# EXPLICITLY BLOCK NON-FEATURE COLUMNS
# ------------------------------------------------

blocked = {
    "target",
    "target_label",
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct",
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end"
}

# Current-window numeric columns only
numeric_cols = df_c.select_dtypes(
    include=np.number
).columns.tolist()

feature_candidates = [
    c for c in numeric_cols
    if c not in blocked
]

print(
    f"\nCandidate numeric features: "
    f"{len(feature_candidates)}"
)

# ------------------------------------------------
# TARGET-WISE MEDIANS
# ------------------------------------------------

target_medians = (
    df_c
    .groupby("target")[feature_candidates]
    .median()
    .T
)

target_medians.columns = [
    "DOWN",
    "FLAT",
    "UP"
]

print("\n" + "=" * 80)
print("TARGET-WISE FEATURE MEDIANS")
print("=" * 80)

display(
    target_medians.round(3)
)

# ------------------------------------------------
# CORRELATION WITH TARGET
# ------------------------------------------------

corr_rows = []

for col in feature_candidates:

    x = pd.to_numeric(
        df_c[col],
        errors="coerce"
    )

    if x.nunique(dropna=True) < 2:
        continue

    corr = x.corr(
        df_c["target"],
        method="spearman"
    )

    corr_rows.append({
        "feature": col,
        "spearman_abs": abs(corr),
        "spearman": corr
    })

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values(
        "spearman_abs",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TOP FEATURES BY TARGET ASSOCIATION")
print("=" * 80)

display(
    corr_df.head(20).round(4)
)

# ------------------------------------------------
# TARGET-WISE MEAN / MEDIAN DIFFERENCE
# ------------------------------------------------

signal_rows = []

for col in feature_candidates:

    grouped = (
        df_c
        .groupby("target")[col]
        .median()
        .reindex([0, 1, 2])
    )

    if grouped.isna().all():
        continue

    signal_rows.append({
        "feature": col,
        "DOWN_median": grouped.iloc[0],
        "FLAT_median": grouped.iloc[1],
        "UP_median": grouped.iloc[2],
        "max_class_gap": (
            grouped.max() - grouped.min()
        )
    })

signal_df = (
    pd.DataFrame(signal_rows)
    .sort_values(
        "max_class_gap",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("FEATURE CLASS-SEPARATION AUDIT")
print("=" * 80)

display(
    signal_df.head(20).round(3)
)

print("\n" + "=" * 80)
print("AUDIT C COMPLETE")
print("=" * 80)

AUDIT C — FEATURE PREDICTIVE SIGNAL

Candidate numeric features: 25

TARGET-WISE FEATURE MEDIANS


,DOWN,FLAT,UP
gsc_clicks_mean_3m,0.000,0.333,0.000
gsc_clicks_last,0.000,0.000,0.000
gsc_impressions_mean_3m,53.000,325.000,90.667
gsc_impressions_last,36.000,378.000,131.000
gsc_avg_position_mean_3m,8.317,8.185,9.060
gsc_avg_position_last,7.493,7.906,8.570
ga4_total_engagement_sec_mean_3m,0.000,0.000,0.000
ga4_total_engagement_sec_last,0.000,0.000,0.000
sessions_organic_mean_3m,0.000,0.000,0.000
sessions_organic_last,0.000,0.000,0.000



TOP FEATURES BY TARGET ASSOCIATION


,feature,spearman_abs,spearman
0,gsc_avg_position_missing_last,0.1555,-0.1555
1,gsc_impressions_last_30d,0.1193,0.1193
2,gsc_impressions_last,0.1193,0.1193
3,ctr_last,0.1132,0.1132
4,gsc_clicks_last,0.0994,0.0994
5,ctr_mean_3m,0.0881,0.0881
6,gsc_impressions_prev_30d,0.0785,-0.0785
7,gsc_avg_position_last,0.0663,0.0663
8,ga4_total_engagement_sec_mean_3m,0.0647,-0.0647
9,gsc_clicks_mean_3m,0.0618,0.0618



FEATURE CLASS-SEPARATION AUDIT


,feature,DOWN_median,FLAT_median,UP_median,max_class_gap
0,gsc_impressions_last,36.000,378.000,131.000,342.000
1,gsc_impressions_last_30d,36.000,378.000,131.000,342.000
2,gsc_impressions_mean_3m,53.000,325.000,90.667,272.000
3,current_imp_3m,53.000,325.000,90.667,272.000
4,gsc_impressions_prev_30d,37.000,212.000,30.000,182.000
5,gsc_avg_position_last,7.493,7.906,8.570,1.077
6,gsc_avg_position_mean_3m,8.317,8.185,9.060,0.876
7,gsc_clicks_mean_3m,0.000,0.333,0.000,0.333
8,ctr_mean_3m,0.000,0.001,0.000,0.001
9,sessions_organic_mean_3m,0.000,0.000,0.000,0.000



AUDIT C COMPLETE


**BLOCK D — Feature Distribution Drift**

In [12]:
# ================================================================
# AUDIT D — FEATURE DISTRIBUTION DRIFT
# ================================================================

from scipy.stats import ks_2samp

print("=" * 80)
print("AUDIT D — FEATURE DISTRIBUTION DRIFT")
print("=" * 80)

df_d = df_audit.copy()

cutoff = pd.Timestamp("2026-01-01")

train_period = df_d[
    df_d["window_start"] < cutoff
]

test_period = df_d[
    df_d["window_start"] >= cutoff
]

blocked = {
    "target",
    "target_label",
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct",
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end"
}

numeric_cols = df_d.select_dtypes(
    include=np.number
).columns.tolist()

features = [
    c for c in numeric_cols
    if c not in blocked
]

rows = []

for col in features:

    train_values = pd.to_numeric(
        train_period[col],
        errors="coerce"
    ).dropna()

    test_values = pd.to_numeric(
        test_period[col],
        errors="coerce"
    ).dropna()

    if len(train_values) < 20 or len(test_values) < 20:
        continue

    # Limit sample size for speed
    n = min(
        20000,
        len(train_values),
        len(test_values)
    )

    train_sample = train_values.sample(
        n=n,
        random_state=42
    )

    test_sample = test_values.sample(
        n=n,
        random_state=42
    )

    statistic, p_value = ks_2samp(
        train_sample,
        test_sample
    )

    rows.append({
        "feature": col,
        "train_median": train_values.median(),
        "test_median": test_values.median(),
        "median_change_pct": (
            (
                test_values.median()
                - train_values.median()
            )
            /
            (
                abs(train_values.median())
                + 1e-9
            )
            * 100
        ),
        "ks_statistic": statistic,
        "p_value": p_value
    })

drift_df = (
    pd.DataFrame(rows)
    .sort_values(
        "ks_statistic",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TOP FEATURE DISTRIBUTION DRIFT")
print("=" * 80)

display(
    drift_df.head(25).round(4)
)

print("\n" + "=" * 80)
print("AUDIT D COMPLETE")
print("=" * 80)

AUDIT D — FEATURE DISTRIBUTION DRIFT

TOP FEATURE DISTRIBUTION DRIFT


,feature,train_median,test_median,median_change_pct,ks_statistic,p_value
0,sessions_organic_mean_3m,0.0000,0.0000,0.0000,0.1039,0.0000
1,sessions_organic_last,0.0000,0.0000,0.0000,0.1030,0.0000
2,ga4_total_engagement_sec_mean_3m,0.0000,0.0000,0.0000,0.0869,0.0000
3,ga4_total_engagement_sec_last,0.0000,0.0000,0.0000,0.0815,0.0000
4,ctr_mean_3m,0.0000,0.0000,0.0000,0.0686,0.0000
5,gsc_avg_position_missing_mean_3m,0.0000,0.0000,0.0000,0.0608,0.0000
6,current_imp_3m,117.3333,93.3333,-20.4545,0.0598,0.0000
7,gsc_impressions_mean_3m,117.3333,93.3333,-20.4545,0.0598,0.0000
8,ctr_last,0.0000,0.0000,0.0000,0.0563,0.0000
9,gsc_avg_position_mean_3m,8.6904,8.2318,-5.2769,0.0561,0.0000



AUDIT D COMPLETE


**BLOCK E — Simple Rule Baseline**

In [13]:
# ================================================================
# AUDIT E — SIMPLE BASELINE SIGNAL
# ================================================================

print("=" * 80)
print("AUDIT E — SIMPLE TREND BASELINE")
print("=" * 80)

df_e = df_audit.copy()

required = [
    "gsc_impressions_mean_3m",
    "gsc_impressions_last",
    "target"
]

available = [
    c for c in required
    if c in df_e.columns
]

print("\nAvailable trend columns:")
for c in available:
    print(" -", c)

if (
    "gsc_impressions_mean_3m" in df_e.columns
    and
    "gsc_impressions_last" in df_e.columns
):

    mean_imp = pd.to_numeric(
        df_e["gsc_impressions_mean_3m"],
        errors="coerce"
    )

    last_imp = pd.to_numeric(
        df_e["gsc_impressions_last"],
        errors="coerce"
    )

    trend_pct = (
        (last_imp - mean_imp)
        /
        (mean_imp.abs() + 1e-9)
        * 100
    )

    df_e["current_trend_pct"] = trend_pct

    print("\n" + "=" * 80)
    print("CURRENT 90-DAY TREND BY FUTURE TARGET")
    print("=" * 80)

    display(
        df_e
        .groupby("target")["current_trend_pct"]
        .agg([
            "count",
            "median",
            "mean",
            "min",
            "max"
        ])
        .round(2)
    )

    # ------------------------------------------------------------
    # SIMPLE RULE
    #
    # Negative current trend -> DOWN
    # Positive current trend -> UP
    # Otherwise FLAT
    # ------------------------------------------------------------

    simple_pred = np.select(
        [
            trend_pct <= -10,
            trend_pct >= 10
        ],
        [
            0,
            2
        ],
        default=1
    )

    valid = (
        trend_pct.notna()
        &
        df_e["target"].notna()
    )

    simple_accuracy = (
        simple_pred[valid]
        ==
        df_e.loc[valid, "target"].to_numpy()
    ).mean()

    print("\n" + "=" * 80)
    print("SIMPLE TREND BASELINE")
    print("=" * 80)

    print(
        f"Accuracy: "
        f"{simple_accuracy*100:.2f}%"
    )

else:

    print(
        "\nRequired impression trend columns "
        "are not available."
    )

print("\n" + "=" * 80)
print("AUDIT E COMPLETE")
print("=" * 80)

AUDIT E — SIMPLE TREND BASELINE

Available trend columns:
 - gsc_impressions_mean_3m
 - gsc_impressions_last
 - target

CURRENT 90-DAY TREND BY FUTURE TARGET


,count,median,mean,min,max
target,,,,,
0,206390,-15.31,-5.92,-100.0,200.0
1,156653,16.67,23.06,-100.0,200.0
2,263793,44.06,50.76,-100.0,200.0



SIMPLE TREND BASELINE
Accuracy: 51.94%

AUDIT E COMPLETE


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Dataset Standard: Same X_test and y_test

Models & Baselines:

Early Drop baseline

Logistic Regression

Random Forest

Metrics & Analysis:

Precision / Recall / F1

True Decay catch rate

Rank correlation with low-impression baseline

Action-threshold comparison

Confusion matrices

**BLOCK 5.4A — RANDOM FOREST TRAINING**

In [7]:
# ============================================================
# BLOCK 5.4A — RANDOM FOREST TRAINING
# ============================================================

from sklearn.ensemble import RandomForestClassifier

print("=" * 80)
print("BLOCK 5.4A — RANDOM FOREST TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# 1. LIGHTWEIGHT RANDOM FOREST
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 2. TRAIN
# ------------------------------------------------------------

print("\nTraining Random Forest...")

rf_model.fit(
    X_train,
    y_train
)

print("✓ Random Forest training complete.")

# ------------------------------------------------------------
# 3. TRAINING INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING INFORMATION")
print("=" * 80)

print(f"Training rows : {len(X_train):,}")
print(f"Features      : {X_train.shape[1]:,}")
print(f"Trees         : {rf_model.n_estimators}")
print(f"Max depth     : {rf_model.max_depth}")
print(f"Min leaf      : {rf_model.min_samples_leaf}")
print(f"CPU cores     : All available")

print("\n✓ BLOCK 5.4A COMPLETE")

BLOCK 5.4A — RANDOM FOREST TRAINING

Training Random Forest...
✓ Random Forest training complete.

TRAINING INFORMATION
Training rows : 486,894
Features      : 26
Trees         : 100
Max depth     : 15
Min leaf      : 2
CPU cores     : All available

✓ BLOCK 5.4A COMPLETE


**BLOCK 5.4B — RANDOM FOREST TESTING + COMPLETE EVALUATION**

In [8]:
# ============================================================
# BLOCK 5.4B — RANDOM FOREST TESTING + EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 5.4B — RANDOM FOREST TESTING + EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. TEST PREDICTIONS
# ------------------------------------------------------------

print("\nGenerating predictions...")

y_pred_rf = rf_model.predict(
    X_test
)

print("✓ Predictions generated.")

# ============================================================
# 2. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2]
)

# ------------------------------------------------------------
# IMPORTANT:
#
# Rows    = ACTUAL
# Columns = PREDICTED
#
#          Pred
#          D    F    U
# Actual
# D
# F
# U
# ------------------------------------------------------------

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual DOWN",
        "Actual FLAT",
        "Actual UP"
    ],
    columns=[
        "Pred DOWN",
        "Pred FLAT",
        "Pred UP"
    ]
)

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

display(cm_df)

# ============================================================
# 3. CLASS-BY-CLASS TP / FP / FN / ACTUAL
# ============================================================

classes = [
    0,
    1,
    2
]

class_names = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

class_results = []

total_samples = len(y_test)

for cls in classes:

    # True Positive
    TP = cm[cls, cls]

    # False Positive
    FP = (
        cm[:, cls].sum()
        - TP
    )

    # False Negative
    FN = (
        cm[cls, :].sum()
        - TP
    )

    # Actual number of this class
    actual_count = cm[cls, :].sum()

    # Predicted number of this class
    predicted_count = cm[:, cls].sum()

    # Percentage of actual class correctly predicted
    actual_recall_pct = (
        TP / actual_count * 100
        if actual_count > 0
        else 0
    )

    # Percentage of model predictions that were correct
    prediction_precision_pct = (
        TP / predicted_count * 100
        if predicted_count > 0
        else 0
    )

    class_results.append({
        "Class": class_names[cls],
        "Actual": actual_count,
        "Predicted": predicted_count,
        "Correct (TP)": TP,
        "Wrong (FP)": FP,
        "Missed (FN)": FN,
        "Recall %": actual_recall_pct,
        "Precision %": prediction_precision_pct
    })

class_results_df = pd.DataFrame(
    class_results
)

class_results_df[
    [
        "Actual",
        "Predicted",
        "Correct (TP)",
        "Wrong (FP)",
        "Missed (FN)",
        "Recall %",
        "Precision %"
    ]
] = class_results_df[
    [
        "Actual",
        "Predicted",
        "Correct (TP)",
        "Wrong (FP)",
        "Missed (FN)",
        "Recall %",
        "Precision %"
    ]
].copy()

class_results_df["Recall %"] = (
    class_results_df["Recall %"]
    .round(2)
)

class_results_df["Precision %"] = (
    class_results_df["Precision %"]
    .round(2)
)

print("\n" + "=" * 80)
print("CLASS-BY-CLASS PREDICTION ANALYSIS")
print("=" * 80)

display(
    class_results_df
)

# ============================================================
# 4. METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred_rf
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred_rf
)

precision_macro = precision_score(
    y_test,
    y_pred_rf,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_test,
    y_pred_rf,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_test,
    y_pred_rf,
    average="macro",
    zero_division=0
)

precision_weighted = precision_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

recall_weighted = recall_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

f1_weighted = f1_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

# ============================================================
# 5. OVERALL EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("RANDOM FOREST OVERALL EVALUATION")
print("=" * 80)

print(
    f"Accuracy           : {accuracy*100:.2f}%"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy*100:.2f}%"
)

print(
    f"Macro Precision    : {precision_macro*100:.2f}%"
)

print(
    f"Macro Recall       : {recall_macro*100:.2f}%"
)

print(
    f"Macro F1 Score     : {f1_macro*100:.2f}%"
)

print(
    f"Weighted Precision : {precision_weighted*100:.2f}%"
)

print(
    f"Weighted Recall    : {recall_weighted*100:.2f}%"
)

print(
    f"Weighted F1 Score  : {f1_weighted*100:.2f}%"
)

# ============================================================
# 6. PER-CLASS METRICS
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS EVALUATION")
print("=" * 80)

precision_per_class = precision_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

recall_per_class = recall_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

f1_per_class = f1_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

per_class_metrics = pd.DataFrame({
    "Class": [
        "DOWN",
        "FLAT",
        "UP"
    ],
    "Precision %": (
        precision_per_class * 100
    ).round(2),
    "Recall %": (
        recall_per_class * 100
    ).round(2),
    "F1 Score %": (
        f1_per_class * 100
    ).round(2)
})

display(
    per_class_metrics
)

# ============================================================
# 7. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_pred_rf,
        labels=[0, 1, 2],
        target_names=[
            "DOWN",
            "FLAT",
            "UP"
        ],
        digits=4,
        zero_division=0
    )
)

# ============================================================
# 8. SIMPLE HUMAN-READABLE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HUMAN-READABLE RESULTS")
print("=" * 80)

for _, row in class_results_df.iterrows():

    print(
        f"\n{row['Class']}"
    )

    print(
        f"  Actual pages      : "
        f"{int(row['Actual']):,}"
    )

    print(
        f"  Predicted pages   : "
        f"{int(row['Predicted']):,}"
    )

    print(
        f"  Correct prediction: "
        f"{int(row['Correct (TP)']):,}"
        f" ({row['Recall %']:.2f}% of actual)"
    )

    print(
        f"  Precision         : "
        f"{row['Precision %']:.2f}%"
    )

    print(
        f"  Missed            : "
        f"{int(row['Missed (FN)']):,}"
    )

# ============================================================
# 9. FINAL VERDICT
# ============================================================

print("\n" + "=" * 80)
print("FINAL RANDOM FOREST RESULT")
print("=" * 80)

print(
    f"Accuracy          : {accuracy*100:.2f}%"
)

print(
    f"Balanced Accuracy : {balanced_accuracy*100:.2f}%"
)

print(
    f"Macro F1          : {f1_macro*100:.2f}%"
)

print(
    f"Weighted F1       : {f1_weighted*100:.2f}%"
)

print("\n✓ BLOCK 5.4B COMPLETE")

BLOCK 5.4B — RANDOM FOREST TESTING + EVALUATION

Generating predictions...
✓ Predictions generated.

CONFUSION MATRIX


,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,25762,8404,40088
Actual FLAT,7194,5382,24647
Actual UP,5182,1150,22133



CLASS-BY-CLASS PREDICTION ANALYSIS


,Class,Actual,Predicted,Correct (TP),Wrong (FP),Missed (FN),Recall %,Precision %
0,DOWN,74254,38138,25762,12376,48492,34.69,67.55
1,FLAT,37223,14936,5382,9554,31841,14.46,36.03
2,UP,28465,86868,22133,64735,6332,77.76,25.48



RANDOM FOREST OVERALL EVALUATION
Accuracy           : 38.07%
Balanced Accuracy  : 42.30%
Macro Precision    : 43.02%
Macro Recall       : 42.30%
Macro F1 Score     : 34.95%
Weighted Precision : 50.61%
Weighted Recall    : 38.07%
Weighted F1 Score  : 37.62%

PER-CLASS EVALUATION


,Class,Precision %,Recall %,F1 Score %
0,DOWN,67.55,34.69,45.84
1,FLAT,36.03,14.46,20.64
2,UP,25.48,77.76,38.38



CLASSIFICATION REPORT
              precision    recall  f1-score   support

        DOWN     0.6755    0.3469    0.4584     74254
        FLAT     0.3603    0.1446    0.2064     37223
          UP     0.2548    0.7776    0.3838     28465

    accuracy                         0.3807    139942
   macro avg     0.4302    0.4230    0.3495    139942
weighted avg     0.5061    0.3807    0.3762    139942


HUMAN-READABLE RESULTS

DOWN
  Actual pages      : 74,254
  Predicted pages   : 38,138
  Correct prediction: 25,762 (34.69% of actual)
  Precision         : 67.55%
  Missed            : 48,492

FLAT
  Actual pages      : 37,223
  Predicted pages   : 14,936
  Correct prediction: 5,382 (14.46% of actual)
  Precision         : 36.03%
  Missed            : 31,841

UP
  Actual pages      : 28,465
  Predicted pages   : 86,868
  Correct prediction: 22,133 (77.76% of actual)
  Precision         : 25.48%
  Missed            : 6,332

FINAL RANDOM FOREST RESULT
Accuracy          : 38.07%
Balanced Ac

**Random Forest tuning code**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.